In [ ]:
# Cell 1 — Objective: Environment setup (requirements)
# Install ONLY the minimal Python packages used by this notebook.
# - No cloud calls, no secrets, no interactive prompts.
# - Keep it deterministic and quiet.

%pip install --quiet --upgrade pip
%pip install --quiet oci ipywidgets paramiko

# Lightweight presence check for required CLIs (no flags to avoid compatibility issues).
import shutil, sys

def _presence(name: str) -> str:
    p = shutil.which(name)
    return f"{name}: {'FOUND at ' + p if p else 'NOT FOUND'}"

print("Python:", sys.version.split()[0])
print(_presence("terraform"))
print(_presence("kubectl"))
print(_presence("oci"))  # optional; used for kubeconfig convenience if present

print("Cell 1 complete: Environment ready")
print("NEXT: Run Cell 2 — Central variables & credentials (no network calls).")


In [ ]:
# Cell 2 — Objective: Central variables & credentials (single source of truth; no network calls)
# - Centralize ALL configurable values; allow environment overrides where safe.
# - Load OCI config locally; DO NOT print secrets; NO network calls.
# - Create OUTPUT_DIR; define TS_UTC (UTC).
# - Design choice in v1.02: Direct Pod Routing only (pods-as-backends). No UI toggle.

import os
import re
from pathlib import Path
from datetime import datetime, timezone
import oci

# --- OCI config (local read/validate; no network calls) ---
# Required environment (by name; values loaded from ~/.oci/config by default):
# - OCI_CONFIG_FILE (path to oci config; default "~/.oci/config")
# - OCI_PROFILE (profile name; default "DEFAULT")
# Optional explicit overrides (fallback to config if unset):
# - TENANCY_OCID, USER_OCID, FINGERPRINT, REGION
# - OCI_PRIVATE_KEY_PATH (path to private key)
# - OCI_PASSPHRASE / OCI_PRIVATE_KEY_PASSPHRASE (key passphrase if set)

OCI_CONFIG_FILE = os.path.expanduser(os.environ.get("OCI_CONFIG_FILE", "~/.oci/config"))
OCI_PROFILE     = os.environ.get("OCI_PROFILE", "DEFAULT")

_cfg = oci.config.from_file(file_location=OCI_CONFIG_FILE, profile_name=OCI_PROFILE)
oci.config.validate_config(_cfg)  # local validation only

TENANCY_OCID = os.environ.get("TENANCY_OCID", _cfg["tenancy"])
USER_OCID    = os.environ.get("USER_OCID",    _cfg["user"])
FINGERPRINT  = os.environ.get("FINGERPRINT",  _cfg["fingerprint"])

OCI_PRIVATE_KEY_PATH   = os.path.expanduser(os.environ.get("OCI_PRIVATE_KEY_PATH", _cfg.get("key_file", "")))
PRIVATE_KEY_PASSPHRASE = os.environ.get("OCI_PASSPHRASE", os.environ.get("OCI_PRIVATE_KEY_PASSPHRASE", _cfg.get("pass_phrase", "")))
REGION                 = os.environ.get("REGION", _cfg["region"])

# Fail fast on required OCI private key file presence (do not print key content)
if not OCI_PRIVATE_KEY_PATH or not Path(OCI_PRIVATE_KEY_PATH).exists():
    raise FileNotFoundError(f"OCI private key not found. Check OCI_CONFIG_FILE/PROFILE or set OCI_PRIVATE_KEY_PATH. Current: {OCI_PRIVATE_KEY_PATH or '(unset)'}")

# --- Deployment mode & namespace ---
# Keep both/generators-only/oke-only to preserve existing flows for infra & generators orchestration.
DEPLOY_MODE = os.environ.get("DEPLOY_MODE", "both").strip().lower()  # "generators-only" | "oke-only" | "both"
WORK_NS     = os.environ.get("WORK_NS", "cpstest").strip()

# Derived enables from DEPLOY_MODE (used later cells; no calls here)
ENABLE_OKE        = DEPLOY_MODE in ("oke-only", "both")
ENABLE_GENERATORS = DEPLOY_MODE in ("generators-only", "both")

# --- One-VCN CIDRs (UI-editable defaults) ---
CP_CIDR              = os.environ.get("CP_CIDR",             "10.0.1.0/24")
PUB_LB_CIDR          = os.environ.get("PUB_LB_CIDR",         "10.0.2.0/24")
WORKERS_PRIVATE_CIDR = os.environ.get("WORKERS_PRIVATE_CIDR","10.0.4.0/22")  # sized for nodes
PODS_CIDR            = os.environ.get("PODS_CIDR",           "10.0.64.0/18") # sized for pods
GENS_PUB_CIDR        = os.environ.get("GENS_PUB_CIDR",       "10.0.20.0/24")

# --- OKE (Backends only) ---
KUBERNETES_VERSION      = os.environ.get("KUBERNETES_VERSION", "v1.34.2")
SSH_PUBLIC_KEY_PATH     = os.path.expanduser(os.environ.get("SSH_PUBLIC_KEY_PATH", "~/.ssh/id_rsa.pub"))
BACKEND_NODE_SHAPE      = os.environ.get("BACKEND_NODE_SHAPE", "VM.Standard.E5.Flex")
BACKEND_OCPUS           = float(os.environ.get("BACKEND_OCPUS", "16"))
BACKEND_MEMORY_GB       = float(os.environ.get("BACKEND_MEMORY_GB", "64"))
MAX_PODS_PER_NODE       = int(os.environ.get("MAX_PODS_PER_NODE", "31"))
BACKEND_POOL_SIZE       = int(os.environ.get("BACKEND_POOL_SIZE", "1"))
NODE_IMAGE_OCID_BACKEND = os.environ.get("NODE_IMAGE_OCID_BACKEND", "").strip() or None  # required later in Cell 4 validations

# Optional UI-persistence for image filter (dynamic in Cell 3; persisted here if set)
ORACLE_LINUX_MAJOR      = os.environ.get("ORACLE_LINUX_MAJOR", "any").strip().lower()  # "any" | "7" | "8" | "9" | "10" | ...

# --- Generators (standalone compute) ---
GENERATOR_COUNT     = int(os.environ.get("GENERATOR_COUNT", "4"))
GENERATOR_SHAPE     = os.environ.get("GENERATOR_SHAPE", "VM.Standard.E5.Flex")
GENERATOR_OCPUS     = float(os.environ.get("GENERATOR_OCPUS", "8"))
GENERATOR_MEMORY_GB = float(os.environ.get("GENERATOR_MEMORY_GB", "32"))
ENABLE_IPV6_ON_GENERATORS = os.environ.get("ENABLE_IPV6_ON_GENERATORS", "false").strip().lower() in ("1","true","yes","y")

# SSH ingress CIDRs for generators (centralized here for later terraform)
SSH_ALLOWED_CIDR    = os.environ.get("SSH_ALLOWED_CIDR", "0.0.0.0/0").strip()
SSH_ALLOWED_V6_CIDR = os.environ.get("SSH_ALLOWED_V6_CIDR", "::/0").strip()

# SSH keys for generators (validated later by UI / terraform)
_default_pub  = Path(os.path.expanduser("~/.ssh/id_rsa.pub"))
_default_priv = Path(os.path.expanduser("~/.ssh/id_rsa"))
GEN_SSH_PUBLIC_KEY_PATH  = os.path.expanduser(os.environ.get("GEN_SSH_PUBLIC_KEY_PATH",  str(_default_pub  if _default_pub.exists()  else "")))
GEN_SSH_PRIVATE_KEY_PATH = os.path.expanduser(os.environ.get("GEN_SSH_PRIVATE_KEY_PATH", str(_default_priv if _default_priv.exists() else "")))

# --- Locust & Targets (used only when generators enabled) ---
TEST_MODE                 = os.environ.get("TEST_MODE", "cps").strip().lower()  # "cps" | "throughput"
LOCUST_WAIT_TIME_SEC      = float(os.environ.get("LOCUST_WAIT_TIME_SEC", "1.0"))
LOCUST_CONNECT_TIMEOUT_MS = int(os.environ.get("LOCUST_CONNECT_TIMEOUT_MS", "8000"))
LOCUST_READ_TIMEOUT_MS    = int(os.environ.get("LOCUST_READ_TIMEOUT_MS",  "15000"))
LOCUST_VERIFY_TLS         = os.environ.get("LOCUST_VERIFY_TLS", "false").strip().lower() in ("1","true","yes","y")
UI_WEB_PORT               = int(os.environ.get("UI_WEB_PORT", "8089"))
EXTERNAL_TARGETS_TEXT     = os.environ.get("EXTERNAL_TARGETS_TEXT", "").strip()  # normalized in Cell 3

# Paths and endpoints used by generator runtime (kept in SSOT here)
LOCUST_WORKDIR           = os.path.abspath(os.environ.get("LOCUST_WORKDIR", "/home/opc/locustwork"))
HEALTH_ENDPOINT_PATH     = os.environ.get("HEALTH_ENDPOINT_PATH", "/healthz")
THROUGHPUT_ENDPOINT_PATH = os.environ.get("THROUGHPUT_ENDPOINT_PATH", "/payload_100k")

# --- CPU → workers policy (authoritative) ---
# Canonical vars only:
#   WORKERS_PER_HOST: "auto" or "<int-string>"
#   CPU_RESERVE: int (only for auto)
#   MIN_WORKERS_PER_HOST: int
#   MAX_WORKERS_PER_HOST: int
WORKERS_PER_HOST      = os.environ.get("WORKERS_PER_HOST", "auto").strip().lower()
CPU_RESERVE           = int(os.environ.get("CPU_RESERVE", "1"))
MIN_WORKERS_PER_HOST  = int(os.environ.get("MIN_WORKERS_PER_HOST", "1"))
MAX_WORKERS_PER_HOST  = int(os.environ.get("MAX_WORKERS_PER_HOST", "32"))

# --- TLS for Service LB (used in Cell 6 to create ssl-certificate-secret) ---
# Force RSA (matches OCI LB default cipher policy). Do not inherit previous env override.
TLS_KEY_ALGO    = "rsa"                               # "rsa" | "ecdsa"
TLS_ECDSA_CURVE = os.environ.get("TLS_ECDSA_CURVE", "prime256v1").strip().lower()  # used only if TLS_KEY_ALGO=ecdsa
# Persist to environment for downstream cells
os.environ["TLS_KEY_ALGO"] = TLS_KEY_ALGO
os.environ["TLS_ECDSA_CURVE"] = TLS_ECDSA_CURVE

# --- Execution toggles (gated steps) ---
RUN_INFRA_APPLY       = os.environ.get("RUN_INFRA_APPLY", "true").strip().lower() in ("1","true","yes","y")
RUN_K8S_DEPLOY        = os.environ.get("RUN_K8S_DEPLOY", "true").strip().lower() in ("1","true","yes","y")
RUN_GENERATORS_PREP   = os.environ.get("RUN_GENERATORS_PREP", "true").strip().lower() in ("1","true","yes","y")
RUN_LOCUST            = os.environ.get("RUN_LOCUST", "false").strip().lower() in ("1","true","yes","y")

DESTROY_GENS_ON_APPLY    = os.environ.get("DESTROY_GENS_ON_APPLY", "false").strip().lower() in ("1","true","yes","y")
DESTROY_OKE_ON_APPLY     = os.environ.get("DESTROY_OKE_ON_APPLY",  "false").strip().lower() in ("1","true","yes","y")
DESTROY_NETWORK_ON_APPLY = os.environ.get("DESTROY_NETWORK_ON_APPLY", "false").strip().lower() in ("1","true","yes","y")

# --- Direct Pod Routing: pods-as-backends (v1.02 design choice) ---
# Hard-set (no UI toggle): Service backends = pods (direct pod routing)
SERVICE_BACKEND_MODE = "pods"
POD_BACKEND_PORT     = int(os.environ.get("POD_BACKEND_PORT", "30080"))  # stable, referenced in Cell 7
os.environ["SERVICE_BACKEND_MODE"] = SERVICE_BACKEND_MODE
os.environ["POD_BACKEND_PORT"]     = str(POD_BACKEND_PORT)

# --- Node pool label (for backend scheduling; SSOT, UI-editable in Cell 3) ---
# Defaults align with Cell 7 nodeSelector: app.role=backend
NODE_LABEL_KEY   = os.environ.get("NODE_LABEL_KEY", "app.role").strip()
NODE_LABEL_VALUE = os.environ.get("NODE_LABEL_VALUE", "backend").strip()

def _valid_dns_subdomain(prefix: str) -> bool:
    # k8s DNS-1123 subdomain: lowercase alnum + '-', '.', start/end alnum; ≤253 total; each label ≤63
    if not prefix or len(prefix) > 253:
        return False
    if prefix.lower() != prefix:
        return False
    parts = prefix.split(".")
    lbl_re = re.compile(r"^[a-z0-9]([a-z0-9-]{0,61}[a-z0-9])?$")
    return all(1 <= len(p) <= 63 and lbl_re.match(p) for p in parts)

def _valid_label_name(name: str) -> bool:
    # 1–63 chars, alnum start/end, middle can include '-', '_', '.'
    return bool(re.match(r"^[A-Za-z0-9]([A-Za-z0-9._-]{0,61}[A-Za-z0-9])?$", name))

def _valid_label_value(val: str) -> bool:
    # k8s values may be empty; we require non-empty here for determinism (adjust in UI if needed)
    return bool(re.match(r"^[A-Za-z0-9]([A-Za-z0-9._-]{0,61}[A-Za-z0-9])?$", val))

# Validate key (optional prefix/name syntax)
if "/" in NODE_LABEL_KEY:
    pref, nm = NODE_LABEL_KEY.split("/", 1)
    if not (_valid_dns_subdomain(pref) and _valid_label_name(nm)):
        raise ValueError(f"Invalid NODE_LABEL_KEY. Expect '<dns-prefix>/<name>' or '<name>'. Current: {NODE_LABEL_KEY}")
else:
    if not _valid_label_name(NODE_LABEL_KEY):
        raise ValueError(f"Invalid NODE_LABEL_KEY name segment. Current: {NODE_LABEL_KEY}")

if not _valid_label_value(NODE_LABEL_VALUE):
    raise ValueError(f"Invalid NODE_LABEL_VALUE. Must be 1–63 chars, alnum start/end; may contain '-', '_', '.'. Current: {NODE_LABEL_VALUE}")

# Persist to environment for downstream cells (Cell 3 UI + Cell 4 Terraform wiring)
os.environ["NODE_LABEL_KEY"] = NODE_LABEL_KEY
os.environ["NODE_LABEL_VALUE"] = NODE_LABEL_VALUE

# --- Placeholders for selections made in Cell 3 (kept here for clarity; set by UI) ---
SELECTED_REGION = None
COMPARTMENT_ID  = ""
AD_A            = ""

# --- Output directory & UTC timestamp ---
OUTPUT_DIR = os.path.abspath(os.environ.get("OUTPUT_DIR", "./results"))
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
TS_UTC = os.environ.get("TS_UTC", "") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
os.environ["TS_UTC"] = TS_UTC  # persist for later cells

# Minimal, non-sensitive confirmation
print(f"Cell 2 ready: {OCI_CONFIG_FILE} [{OCI_PROFILE}] — region={REGION}, mode={DEPLOY_MODE}, ns={WORK_NS}")
print(f"Direct Pod Routing: backend_mode={SERVICE_BACKEND_MODE}, pod_backend_port={POD_BACKEND_PORT}")
print(f"TLS: key_algo={TLS_KEY_ALGO}" + (f", ecdsa_curve={TLS_ECDSA_CURVE}" if TLS_KEY_ALGO == 'ecdsa' else ""))
print(f"Node pool label (backend scheduling): {NODE_LABEL_KEY}={NODE_LABEL_VALUE}")
print("NEXT: Run Cell 3 — Configuration UI (pods-as-backends is fixed; background validation on Apply).")


In [ ]:
# Cell 3 — Objective: Configuration UI (left‑aligned) + Background validation + Mode gating
# - Location (Region, Compartment, AD)
# - OKE (Backends only):
#   * OKE K8s version from CE cluster options
#   * Backend shape + Flex OCPUs/Mem (only for *.Flex)
#   * Oracle Linux major filter (Any/OL10/OL9/OL8/OL7, discovered dynamically)
#   * OKE Worker image override (CE sources → strict OKE-only → version/arch/GPU/OL-major → shape-compat)
#   * SSH public key (for OKE)
#   * Max pods/node + Backend nodes
#   * NEW: Node pool label key/value (used by Cell 4 to label all nodes on creation)
# - Generators (Compute):
#   * Shape filter + shape dropdown (+ Flex OCPUs/Mem for *.Flex)
#   * Generator image (Oracle Linux; shape-compat validated)
#   * SSH pub (GEN) + SSH priv (GEN)
#   * Workers/host CPU policy: mode=auto/fixed, fixed count, CPU reserve, min/max
#   * Count + IPv6 toggle
# - One‑VCN CIDRs + Locust & Targets + Execution toggles
# - Capacity Estimator UI is removed; capacity is validated in the background on Apply and shown only on error.
# - v1.02 design choice: Direct Pod Routing is fixed (pods-as-backends). No UI toggle.

import os
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML
import oci
import re
import ipaddress

# ----- CSS (left-aligned, clean layout) -----
display(HTML("""
<style>
  .nb3-container { max-width: 1100px; margin: 0 auto; }
  .nb3-section { border: 1px solid #e0e0e0; background: #f8f9fb; padding: 12px; margin: 8px 0; border-left: 4px solid #d2d7e5; }
  .nb3-title { font-weight: 600; margin: 0 0 6px 0; text-align: left; }
  .nb3-lbl { white-space: nowrap; font-weight: 500; text-align: left; }
  .issues-strip { background: #fdecea; color: #ba1a1a; padding: 6px 10px; border: 1px solid #f5c2c7; border-radius: 6px; font-size: 13px; margin: 0 0 6px 0; }
  .nb3-small { color:#555; font-size:12px; }
  .nb3-pre { font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, "Courier New"; white-space: pre; margin: 0; }
  .info-strip { background: #e8f4fd; color: #0b5394; padding: 6px 10px; border: 1px solid #a4c2f4; border-radius: 6px; font-size: 13px; margin: 0 0 6px 0; }
</style>
"""))

# ----- Small helpers -----
def _usable_ips(cidr: str) -> int:
    try:
        net = ipaddress.ip_network(cidr, strict=True)
        return max(0, net.num_addresses - 3)  # OCI reserves 3
    except Exception:
        return -1

def _set_dd_options_preserve(dd: widgets.Dropdown, pairs: list, prefer: str = None):
    old_val = dd.value
    dd.options = pairs if pairs else [("None", "")]
    new_vals = [v for _, v in dd.options]
    if old_val in new_vals:
        dd.value = old_val
    elif prefer in new_vals:
        dd.value = prefer
    else:
        dd.value = new_vals[0] if new_vals else ""

def row(label_text: str, control: widgets.Widget, label_width="220px") -> widgets.HBox:
    lbl = widgets.HTML(f"<span class='nb3-lbl'>{label_text}</span>",
                       layout=widgets.Layout(width=label_width, min_width=label_width))
    if hasattr(control, "layout"):
        control.layout.flex = "1 1 auto"
        control.layout.width = "auto"
    return widgets.HBox([lbl, control],
                        layout=widgets.Layout(width="100%", justify_content="flex-start", align_items="center", gap="12px"))

def section(title: str, *children: widgets.Widget) -> widgets.VBox:
    return widgets.VBox([widgets.HTML(f"<div class='nb3-title'>{title}</div>"), *children],
                        layout=widgets.Layout(width="100%", align_items="stretch"),
                        _dom_classes=['nb3-section'])

# Image name/parser helpers
_re_ol  = re.compile(r'(?:Oracle[- ]Linux|Oracle[-]?Linux|OL)[-_ ](?P<major>\d+)(?:\.(?P<minor>\d+))?', re.IGNORECASE)
_re_oke = re.compile(r'-OKE-(?:v)?(?P<oke>\d+\.\d+(?:\.\d+)?)\b', re.IGNORECASE)
_re_oke_any = re.compile(r'-OKE-', re.IGNORECASE)
_re_date = re.compile(r'-(?P<date>\d{4}\.\d{2}\.\d{2})-', re.IGNORECASE)

def _parse_image_traits(name: str):
    ol_major = None; ol_minor = None; oke = None; date = None
    m = _re_ol.search(name or "")
    if m:
        try:
            ol_major = int(m.group('major')) if m.group('major') else None
            ol_minor = int(m.group('minor')) if m.group('minor') else None
        except Exception:
            pass
    k = _re_oke.search(name or "")
    if k:
        oke = k.group('oke')
    d = _re_date.search(name or "")
    if d:
        date = d.group('date')
    lower = (name or "").lower()
    is_arm = ('-aarch64-' in lower)
    is_gpu = ('-gpu-' in lower)
    has_oke = bool(_re_oke_any.search(name or ""))
    return {"ol_major": ol_major, "ol_minor": ol_minor, "oke_version": oke, "is_arm": is_arm, "is_gpu": is_gpu, "date": date, "has_oke": has_oke}

def _shape_arch_flags(shape_name: str):
    s = (shape_name or "").lower()
    is_arm = any(tok in s for tok in ['.a1.', '.a2.', 'aarch', 'arm', 'ampere'])
    is_gpu = ('gpu' in s)
    return is_arm, is_gpu

def _norm_k8s_version(ver: str):
    v = (ver or "").strip()
    if v.startswith('v'): v = v[1:]
    parts = v.split('.')
    full = v if len(parts) >= 2 else v
    line = '.'.join(parts[:2]) if len(parts) >= 2 else v
    return full, line

# ----- Label validation helpers (UI-time checks; mirrors Cell 2 constraints) -----
def _valid_dns_subdomain(prefix: str) -> bool:
    if not prefix or len(prefix) > 253:
        return False
    if prefix.lower() != prefix:
        return False
    parts = prefix.split(".")
    lbl_re = re.compile(r"^[a-z0-9]([a-z0-9-]{0,61}[a-z0-9])?$")
    return all(1 <= len(p) <= 63 and lbl_re.match(p) for p in parts)

def _valid_label_name(name: str) -> bool:
    return bool(re.match(r"^[A-Za-z0-9]([A-Za-z0-9._-]{0,61}[A-Za-z0-9])?$", name))

def _valid_label_value(val: str) -> bool:
    return bool(re.match(r"^[A-Za-z0-9]([A-Za-z0-9._-]{0,61}[A-Za-z0-9])?$", val))

def _validate_label_fields(key: str, val: str) -> str:
    k = (key or "").strip()
    v = (val or "").strip()
    if not k or not v:
        return "Node pool label key/value cannot be empty."
    if "/" in k:
        pref, nm = k.split("/", 1)
        if not (_valid_dns_subdomain(pref) and _valid_label_name(nm)):
            return "Invalid label key. Expected '<dns-prefix>/<name>' with DNS-1123 prefix and 1–63 char name."
    else:
        if not _valid_label_name(k):
            return "Invalid label key. Name must be 1–63 chars, alnum start/end, may contain '-', '_', '.'."
    if not _valid_label_value(v):
        return "Invalid label value. Must be 1–63 chars, alnum start/end; may contain '-', '_', '.'."
    return ""

# ----- OCI config + clients (region-scoped) -----
OCI_CONFIG_FILE = os.path.expanduser(os.environ.get("OCI_CONFIG_FILE", "~/.oci/config"))
OCI_PROFILE     = os.environ.get("OCI_PROFILE", "DEFAULT")
_cfg = oci.config.from_file(file_location=OCI_CONFIG_FILE, profile_name=OCI_PROFILE)
TENANCY_OCID = _cfg["tenancy"]

def cfg_for_region(region_name):
    c = dict(_cfg)
    c["region"] = region_name
    return c

def build_signer_from_cfg(cfg=_cfg):
    key_path = os.path.expanduser(cfg.get("key_file", ""))
    if not key_path or not Path(key_path).exists():
        raise FileNotFoundError(f"OCI private key not found at: {key_path}. Check OCI config.")
    return oci.signer.Signer(
        tenancy=cfg["tenancy"], user=cfg["user"], fingerprint=cfg["fingerprint"],
        private_key_file_location=key_path, pass_phrase=cfg.get("pass_phrase")
    )

def identity_client_for_current():
    cfg_r = cfg_for_region(region_dd.value)
    return oci.identity.IdentityClient(config=cfg_r, signer=build_signer_from_cfg(cfg_r))

def compute_client_for_current():
    cfg_r = cfg_for_region(region_dd.value)
    return oci.core.compute_client.ComputeClient(config=cfg_r, signer=build_signer_from_cfg(cfg_r)) if hasattr(oci.core, "compute_client") else oci.core.ComputeClient(config=cfg_r, signer=build_signer_from_cfg(cfg_r))

def container_engine_client_for_current():
    cfg_r = cfg_for_region(region_dd.value)
    return oci.container_engine.ContainerEngineClient(config=cfg_r, signer=build_signer_from_cfg(cfg_r))

# ----- Location (Region, Compartment, AD) -----
loc_error = widgets.HTML("")
try:
    idc_base = oci.identity.IdentityClient(config=_cfg, signer=build_signer_from_cfg())
    subs = sorted(idc_base.list_region_subscriptions(TENANCY_OCID).data, key=lambda r: r.region_name)
    region_options = [(r.region_name, r.region_name) for r in subs]
    default_region = os.environ.get("REGION", _cfg["region"])
    if not any(r.region_name == default_region for r in subs):
        default_region = subs[0].region_name
except Exception as e:
    region_options = [("Config error — check OCI config/key_file", "")]
    default_region = ""
    loc_error.value = f"<div class='issues-strip'>Location discovery error: {e}</div>"

region_dd = widgets.Dropdown(options=region_options, value=default_region)
reload_btn = widgets.Button(description="Reload", icon="refresh")
reset_btn  = widgets.Button(description="Reset",  icon="history")

comp_dd = widgets.Dropdown(options=[("Loading compartments…", "")], value="")
ad_dd   = widgets.Dropdown(options=[("Loading ADs…", "")], value="")

def list_compartments(idc):
    comps = oci.pagination.list_call_get_all_results(
        idc.list_compartments, TENANCY_OCID,
        compartment_id_in_subtree=True, access_level="ACCESSIBLE"
    ).data
    comps = [c for c in comps if c.lifecycle_state == "ACTIVE"]
    tenancy = idc.get_tenancy(TENANCY_OCID).data
    tenancy_name = getattr(tenancy, "name", "root-tenancy")
    return [(f"{tenancy_name} (root)", TENANCY_OCID)] + sorted([(c.name + f" ({c.description or 'no-desc'})", c.id) for c in comps], key=lambda t: t[0].lower())

def list_ads(idc):
    ads = oci.pagination.list_call_get_all_results(idc.list_availability_domains, TENANCY_OCID).data
    return sorted([ad.name for ad in ads]) or ["AD-1"]

def refresh_location(_=None):
    try:
        idc = identity_client_for_current()
        comps = list_compartments(idc)
        _set_dd_options_preserve(comp_dd, comps, prefer=TENANCY_OCID)
        ads = list_ads(idc)
        _set_dd_options_preserve(ad_dd, [(n, n) for n in ads], prefer=(ads[0] if ads else ""))
        loc_error.value = ""
    except Exception as e:
        comp_dd.options = [("Error loading compartments", "")]
        comp_dd.value = ""
        ad_dd.options = [("Error loading ADs", "")]
        ad_dd.value = ""
        loc_error.value = f"<div class='issues-strip'>Location discovery error: {e}</div>"

region_dd.observe(refresh_location, names="value")
reload_btn.on_click(refresh_location)
reset_btn.on_click(lambda _: (setattr(region_dd, "value", default_region), refresh_location()))
if default_region:
    refresh_location()

# ----- Mode & Namespace -----
DEPLOY_MODE = os.environ.get("DEPLOY_MODE", "both").strip().lower()
WORK_NS     = os.environ.get("WORK_NS", "cpstest").strip()
mode_dd = widgets.Dropdown(
    options=[("Generators only", "generators-only"), ("OKE only", "oke-only"), ("Both (OKE + Generators)", "both")],
    value=DEPLOY_MODE
)
ns_in = widgets.Text(value=WORK_NS)

# ----- One‑VCN CIDRs -----
cp_cidr_in   = widgets.Text(value=os.environ.get("CP_CIDR", "10.0.1.0/24"))
plb_cidr_in  = widgets.Text(value=os.environ.get("PUB_LB_CIDR", "10.0.2.0/24"))
wkp_cidr_in  = widgets.Text(value=os.environ.get("WORKERS_PRIVATE_CIDR","10.0.4.0/22"))
pods_cidr_in = widgets.Text(value=os.environ.get("PODS_CIDR", "10.0.64.0/18"))
gen_cidr_in  = widgets.Text(value=os.environ.get("GENS_PUB_CIDR","10.0.20.0/24"))

# ----- OKE (Backends only) -----
KUBERNETES_VERSION = os.environ.get("KUBERNETES_VERSION", "v1.34.2")
oke_k8s_dd     = widgets.Dropdown(options=[(KUBERNETES_VERSION, KUBERNETES_VERSION)], value=KUBERNETES_VERSION)
oke_k8s_reload = widgets.Button(description="Reload versions", icon="refresh")
oke_versions_info = widgets.HTML("<span class='nb3-small'></span>")

def refresh_oke_versions(_=None):
    try:
        ce = container_engine_client_for_current()
        resp = ce.get_cluster_options("all")
        data = getattr(resp, "data", resp)
        vers = getattr(data, "available_kubernetes_versions", None) or getattr(data, "kubernetes_versions", None) or []
        seen = set(); ordered = []
        for v in vers:
            if v not in seen:
                seen.add(v); ordered.append(v)
        items = [(v, v) for v in ordered] or [(KUBERNETES_VERSION, KUBERNETES_VERSION)]
        _set_dd_options_preserve(oke_k8s_dd, items, prefer=KUBERNETES_VERSION)
        oke_versions_info.value = f"<span class='nb3-small'>Loaded {len(items)} OKE versions</span>"
    except Exception:
        oke_versions_info.value = "<span class='nb3-small' style='color:#b00020'>Version load error; using current value.</span>"

oke_k8s_reload.on_click(refresh_oke_versions)
region_dd.observe(refresh_oke_versions, names="value")
comp_dd.observe(refresh_oke_versions, names="value")
refresh_oke_versions()

# Backend shape + Flex CPU/Mem
shape_filter_in   = widgets.Text(value="")
shape_loading_lbl = widgets.HTML("<span class='nb3-small'></span>")
b_shape_default   = os.environ.get("BACKEND_NODE_SHAPE","VM.Standard.E5.Flex")
b_shape_dd        = widgets.Dropdown(options=[(b_shape_default, b_shape_default)], value=b_shape_default)
b_ocpus_in        = widgets.BoundedFloatText(value=float(os.environ.get("BACKEND_OCPUS","16")), min=1, max=1024, step=1)
b_mem_in          = widgets.BoundedFloatText(value=float(os.environ.get("BACKEND_MEMORY_GB","64")), min=1, max=4096, step=1)

def _toggle_backend_flex(shape_name: str):
    is_flex = (shape_name or "").endswith(".Flex")
    b_ocpus_in.layout.display = "" if is_flex else "none"
    b_mem_in.layout.display   = "" if is_flex else "none"

# Oracle Linux major selector (dynamic)
ol_major_dd = widgets.Dropdown(options=[("Any","any")], value=os.environ.get("ORACLE_LINUX_MAJOR","any"))

# OKE Worker image override (CE-first; strict OKE-only; version/arch/GPU/OL-major; shape-compat)
b_img_dd        = widgets.Dropdown(options=[("Leave empty for default OKE image", "")], value="")
b_img_selected  = widgets.HTML("<span class='nb3-small'>Selected image: (default)</span>")
b_img_note      = widgets.HTML("<span class='nb3-small'></span>")
b_img_diag      = widgets.HTML("<span class='nb3-small'></span>")

def _update_selected_image_preview(dd: widgets.Dropdown, label_html: widgets.HTML, default_text="(default)"):
    try:
        lab = next((lab for (lab, val) in dd.options if val == dd.value), default_text)
        label_html.value = f"<span class='nb3-small'>Selected image: {lab}</span>"
    except Exception:
        label_html.value = f"<span class='nb3-small'>Selected image: {default_text}</span>"

def refresh_backend_shapes_and_images(*_):
    # Shapes
    try:
        cc = compute_client_for_current()
        shape_loading_lbl.value = "<span class='nb3-small'>loading shapes…</span>"
        shapes = oci.pagination.list_call_get_all_results(cc.list_shapes, TENANCY_OCID).data
        all_names = sorted({s.shape for s in shapes})
        filt = (shape_filter_in.value or "").strip().lower()
        names = [n for n in all_names if (filt in n.lower())] if filt else all_names
        items = [(n, n) for n in names] or [(b_shape_dd.value, b_shape_dd.value)]
        cur = b_shape_dd.value
        _set_dd_options_preserve(b_shape_dd, items, prefer=cur or b_shape_default)
        _toggle_backend_flex(b_shape_dd.value)
        shape_loading_lbl.value = "<span class='nb3-small'></span>"
        # Refresh OKE images after shape updates
        refresh_worker_images()
    except Exception:
        shape_loading_lbl.value = "<span class='nb3-small' style='color:#b00020'>shape load error</span>"

shape_filter_in.observe(refresh_backend_shapes_and_images, names="value")
b_shape_dd.observe(lambda ch: refresh_backend_shapes_and_images(), names="value")

def refresh_worker_images(*_):
    try:
        b_img_note.value = "<span class='nb3-small'></span>"
        b_img_diag.value = "<span class='nb3-small'></span>"

        ce = container_engine_client_for_current()
        cc = compute_client_for_current()

        # CE node pool options (authoritative)
        ce_sources_ok = True
        try:
            opt = ce.get_node_pool_options("all")
            data = getattr(opt, "data", opt)
            sources = getattr(data, "sources", None) or []
        except Exception:
            sources = []
            ce_sources_ok = False

        pool = []
        for src in sources:
            try:
                name = getattr(src, "source_name", None)
                img  = getattr(src, "image_id", None)
                st   = getattr(src, "source_type", None)
                if not (name and img and (st or "").upper() == "IMAGE"):
                    continue
                if "-OKE-" not in name:
                    continue
                pool.append((img, name))
            except Exception:
                continue

        # Fallback to Compute if CE returns nothing (selected comp → root) and keep only names with "-OKE-"
        comp_sel = comp_dd.value or TENANCY_OCID
        def _compute_oke_only(compartment_id) -> list:
            try:
                resp = oci.pagination.list_call_get_all_results(
                    cc.list_images, compartment_id,
                    sort_by="TIMECREATED", sort_order="DESC"
                )
                out = []
                for im in resp.data[:2000]:
                    nm = im.display_name or ""
                    if "-OKE-" in nm:
                        out.append((im.id, nm))
                return out
            except Exception:
                return []

        comp_pool = []; root_pool = []
        if not pool:
            comp_pool = _compute_oke_only(comp_sel)
            if not comp_pool:
                root_pool = _compute_oke_only(TENANCY_OCID)
        initial = pool if pool else (comp_pool if comp_pool else root_pool)

        if not initial:
            src_stat = "ok" if ce_sources_ok else "error"
            b_img_diag.value = f"<div class='issues-strip'>No OKE images found. Region={region_dd.value}. CE sources={src_stat}. Compute(selected)={len(comp_pool)} Compute(root)={len(root_pool)}. If both are 0, verify IAM policy for images.</div>"
            _set_dd_options_preserve(b_img_dd, [("Leave empty for default OKE image", "")], prefer="")
            _update_selected_image_preview(b_img_dd, b_img_selected)
            return

        # Filters
        ver_full, ver_line = _norm_k8s_version(oke_k8s_dd.value or "")
        is_arm_shape, is_gpu_shape = _shape_arch_flags(b_shape_dd.value or "")

        # Dynamic OL majors from pool
        majors = set()
        for (_img, nm) in initial:
            tr = _parse_image_traits(nm)
            if tr["ol_major"]:
                majors.add(tr["ol_major"])
        major_opts = [("Any","any")] + [(f"OL{m}", str(m)) for m in sorted(majors, reverse=True)]
        _set_dd_options_preserve(ol_major_dd, major_opts, prefer=ol_major_dd.value if ol_major_dd.value in [v for _,v in major_opts] else "any")
        if ol_major_dd.value != "any" and (not majors or int(ol_major_dd.value) not in majors):
            ol_major_dd.value = "any"
            b_img_note.value = "<span class='nb3-small'>No OKE images for selected OL major; reset to Any.</span>"

        def _accept_by_traits(nm: str) -> bool:
            tr = _parse_image_traits(nm)
            if ol_major_dd.value != "any":
                try:
                    want = int(ol_major_dd.value)
                    if tr["ol_major"] != want:
                        return False
                except Exception:
                    return False
            if is_arm_shape and not tr["is_arm"]:
                return False
            if (not is_arm_shape) and tr["is_arm"]:
                return False
            if is_gpu_shape and not tr["is_gpu"]:
                return False
            if (not is_gpu_shape) and tr["is_gpu"]:
                return False
            return True

        trait_filtered = [(img, nm) for (img, nm) in initial if _accept_by_traits(nm)]

        def _ver_full(nm: str) -> bool:
            tr = _parse_image_traits(nm); return (tr["oke_version"] == ver_full) if ver_full else False
        def _ver_line(nm: str) -> bool:
            tr = _parse_image_traits(nm)
            if not tr["oke_version"] or not ver_line:
                return False
            return tr["oke_version"].split('.')[:2] == ver_line.split('.')[:2]
        def _ver_any(nm: str) -> bool:
            return _parse_image_traits(nm)["oke_version"] is not None

        strict_full = [(img, nm) for (img, nm) in trait_filtered if _ver_full(nm)]
        strict_line = [(img, nm) for (img, nm) in trait_filtered if _ver_line(nm) and (img, nm) not in strict_full]
        relaxed_any = [(img, nm) for (img, nm) in trait_filtered if _ver_any(nm) and (img, nm) not in strict_full and (img, nm) not in strict_line]

        # Shape compatibility validation (accept exact or base for Flex)
        def _is_shape_compatible(image_id: str, shape_name: str) -> bool:
            try:
                resp = oci.pagination.list_call_get_all_results(
                    cc.list_image_shape_compatibility_entries, image_id=image_id
                )
                shapes = {e.shape for e in (resp.data or [])}
                if shape_name in shapes:
                    return True
                if shape_name.endswith(".Flex"):
                    base = shape_name[:-5]
                    if base in shapes:
                        return True
                return False
            except Exception:
                return False

        def _compat_pass(items):
            out = []
            for (img, nm) in items:
                if _is_shape_compatible(img, b_shape_dd.value or ""):
                    out.append((img, nm))
                if len(out) >= 200:
                    break
            return out

        strict_full  = _compat_pass(strict_full)
        strict_line  = _compat_pass(strict_line)
        relaxed_any  = _compat_pass(relaxed_any)

        # Build labeled dropdown (priority: exact → minor → any)
        def _mk_label(nm: str, prefix=""):
            tr = _parse_image_traits(nm)
            tags = []
            if tr["is_arm"]:
                tags.append("ARM")
            if tr["is_gpu"]:
                tags.append("GPU")
            tag_txt = f" [{' '.join(tags)}]" if tags else ""
            date = tr["date"] or ""
            return f"{prefix}{tag_txt} {nm}" + (f" — {date}" if date else "")

        pairs = [("Leave empty for default OKE image", "")]
        used = set()

        if strict_full:
            for (img, nm) in strict_full:
                if img in used: continue
                pairs.append((_mk_label(nm, f"[OKE v{_norm_k8s_version(oke_k8s_dd.value)[0]}] "), img)); used.add(img)
        if strict_line:
            for (img, nm) in strict_line:
                if img in used: continue
                pairs.append((_mk_label(nm, f"[OKE v{_norm_k8s_version(oke_k8s_dd.value)[1]}] "), img)); used.add(img)
        if relaxed_any:
            for (img, nm) in relaxed_any:
                if img in used: continue
                pairs.append((_mk_label(nm, "[OKE] "), img)); used.add(img)

        if len(pairs) == 1:
            ce_stat = "ok" if ce_sources_ok else "error"
            b_img_diag.value = f"<div class='issues-strip'>No OKE images after filters. Region={region_dd.value}. CE sources={ce_stat}. Initial={len(initial)} trait={len(trait_filtered)} full={len(strict_full)} minor={len(strict_line)} any={len(relaxed_any)}. Leaving override empty is recommended.</div>"

        _set_dd_options_preserve(b_img_dd, pairs[:201], prefer=os.environ.get("NODE_IMAGE_OCID_BACKEND","") or "")
        _update_selected_image_preview(b_img_dd, b_img_selected)

    except Exception:
        b_img_diag.value = "<div class='issues-strip'>Image load error. Verify region/credentials.</div>"
        _set_dd_options_preserve(b_img_dd, [("Leave empty for default OKE image", "")], prefer="")
        _update_selected_image_preview(b_img_dd, b_img_selected)

oke_k8s_dd.observe(refresh_worker_images, names="value")
region_dd.observe(refresh_worker_images, names="value")
comp_dd.observe(refresh_worker_images, names="value")
ol_major_dd.observe(refresh_worker_images, names="value")

# Initial load of shapes+images
refresh_backend_shapes_and_images()

# OKE SSH pub key (from ~/.ssh)
ssh_dir = Path(os.path.expanduser("~/.ssh"))
ssh_pub_candidates = sorted([str(f) for f in ssh_dir.iterdir() if f.is_file() and f.name.endswith(".pub")]) if (ssh_dir.exists() and ssh_dir.is_dir()) else []
SSH_PUBLIC_KEY_PATH = os.path.expanduser(os.environ.get("SSH_PUBLIC_KEY_PATH", ssh_pub_candidates[0] if ssh_pub_candidates else ""))
ssh_pub_oke_dd = widgets.Dropdown(options=[(p, p) for p in ssh_pub_candidates] or [("No *.pub in ~/.ssh", "")],
                                  value=(SSH_PUBLIC_KEY_PATH if SSH_PUBLIC_KEY_PATH in ssh_pub_candidates else (ssh_pub_candidates[0] if ssh_pub_candidates else "")))

# Other OKE params
max_pods_in  = widgets.BoundedIntText(value=int(os.environ.get("MAX_PODS_PER_NODE", "31")), min=1, max=512, step=1)
pool_size_in = widgets.BoundedIntText(value=int(os.environ.get("BACKEND_POOL_SIZE", "1")), min=1, max=5000, step=1)

# NEW: Node pool label key/value UI (defaults from Cell 2 env)
node_label_key_in = widgets.Text(value=os.environ.get("NODE_LABEL_KEY", "app.role"))
node_label_val_in = widgets.Text(value=os.environ.get("NODE_LABEL_VALUE", "backend"))
node_label_err    = widgets.HTML("")

def _node_label_validate_live(*_):
    msg = _validate_label_fields(node_label_key_in.value, node_label_val_in.value)
    node_label_err.value = f"<div class='issues-strip'>{msg}</div>" if msg else "<span class='nb3-small'>Label format OK</span>"

node_label_key_in.observe(_node_label_validate_live, names="value")
node_label_val_in.observe(_node_label_validate_live, names="value")
_node_label_validate_live()

# ----- Generators (Compute) -----
GENERATOR_COUNT     = int(os.environ.get("GENERATOR_COUNT", "4"))
GENERATOR_SHAPE     = os.environ.get("GENERATOR_SHAPE", "VM.Standard.E5.Flex")
GENERATOR_OCPUS     = float(os.environ.get("GENERATOR_OCPUS", "8"))
GENERATOR_MEMORY_GB = float(os.environ.get("GENERATOR_MEMORY_GB", "32"))
ENABLE_IPV6_ON_GENERATORS = os.environ.get("ENABLE_IPV6_ON_GENERATORS", "false").strip().lower() in ("1","true","yes","y")

g_shape_filter_in = widgets.Text(value="")
g_shape_loading   = widgets.HTML("<span class='nb3-small'></span>")
g_shape_dd        = widgets.Dropdown(options=[(GENERATOR_SHAPE, GENERATOR_SHAPE)], value=GENERATOR_SHAPE)
g_ocpus_in        = widgets.BoundedFloatText(value=GENERATOR_OCPUS, min=1, max=1024, step=1)
g_mem_in          = widgets.BoundedFloatText(value=GENERATOR_MEMORY_GB, min=1, max=4096, step=1)

def _toggle_gen_flex(shape_name: str):
    is_flex = (shape_name or "").endswith(".Flex")
    g_ocpus_in.layout.display = "" if is_flex else "none"
    g_mem_in.layout.display   = "" if is_flex else "none"

g_img_dd        = widgets.Dropdown(options=[("Select a shape first", "")], value="")
g_img_loading   = widgets.HTML("<span class='nb3-small'></span>")
g_img_selected  = widgets.HTML("<span class='nb3-small'>Selected image: (none)</span>")
g_count_in      = widgets.BoundedIntText(value=GENERATOR_COUNT, min=1, max=10000, step=1)
g_v6_cb         = widgets.Checkbox(value=ENABLE_IPV6_ON_GENERATORS)

# SSH keys (GEN)
ssh_pub_candidates_gen = ssh_pub_candidates
ssh_priv_candidates_gen = sorted([str(f) for f in ssh_dir.iterdir() if f.is_file() and not f.name.endswith(".pub")]) if (ssh_dir.exists() and ssh_dir.is_dir()) else []
GEN_SSH_PUBLIC_KEY_PATH  = os.path.expanduser(os.environ.get("GEN_SSH_PUBLIC_KEY_PATH",  (ssh_pub_candidates_gen[0] if ssh_pub_candidates_gen else "")))
GEN_SSH_PRIVATE_KEY_PATH = os.path.expanduser(os.environ.get("GEN_SSH_PRIVATE_KEY_PATH", (ssh_priv_candidates_gen[0] if ssh_priv_candidates_gen else "")))
ssh_pub_gen_dd  = widgets.Dropdown(options=[(p, p) for p in ssh_pub_candidates_gen]  or [("No *.pub in ~/.ssh", "")],
                                   value=(GEN_SSH_PUBLIC_KEY_PATH if GEN_SSH_PUBLIC_KEY_PATH in ssh_pub_candidates_gen else (ssh_pub_candidates_gen[0] if ssh_pub_candidates_gen else "")))
ssh_priv_gen_dd = widgets.Dropdown(options=[(p, p) for p in ssh_priv_candidates_gen] or [("No private keys in ~/.ssh", "")],
                                   value=(GEN_SSH_PRIVATE_KEY_PATH if GEN_SSH_PRIVATE_KEY_PATH in ssh_priv_candidates_gen else (ssh_priv_candidates_gen[0] if ssh_priv_candidates_gen else "")))

# Workers/host CPU policy (auto/fixed; canonical only)
def _default_mode_from_value(v):
    if str(v).strip().lower() == "auto":
        return "auto"
    try:
        int(str(v).strip())
        return "fixed"
    except Exception:
        return "auto"

workers_per_host_env = os.environ.get("WORKERS_PER_HOST", "auto").strip().lower() or "auto"
workers_per_host_mode_in = widgets.Dropdown(
    options=[("Auto (use OCPUs)", "auto"), ("Fixed count", "fixed")],
    value=_default_mode_from_value(workers_per_host_env),
    description=""
)
fixed_workers_default = 1
if _default_mode_from_value(workers_per_host_env) == "fixed":
    try:
        fixed_workers_default = max(1, int(workers_per_host_env))
    except Exception:
        fixed_workers_default = 1
fixed_workers_in = widgets.BoundedIntText(value=fixed_workers_default, min=1, max=1024, step=1)
cpu_reserve_in   = widgets.BoundedIntText(value=int(os.environ.get("CPU_RESERVE","1")), min=0, max=16, step=1)
min_workers_in   = widgets.BoundedIntText(value=int(os.environ.get("MIN_WORKERS_PER_HOST","1")), min=1, max=1024, step=1)
max_workers_in   = widgets.BoundedIntText(value=int(os.environ.get("MAX_WORKERS_PER_HOST","32")), min=1, max=2048, step=1)

lbl_err_workers = widgets.HTML("")

def _toggle_fixed_inputs(_=None):
    fixed_workers_in.disabled = (workers_per_host_mode_in.value != "fixed")
_toggle_fixed_inputs()
workers_per_host_mode_in.observe(_toggle_fixed_inputs, names="value")

def refresh_gen_shapes_and_images(*_):
    try:
        cc = compute_client_for_current()
        # Shapes
        g_shape_loading.value = "<span class='nb3-small'>loading shapes…</span>"
        shapes = oci.pagination.list_call_get_all_results(cc.list_shapes, TENANCY_OCID).data
        all_names = sorted({s.shape for s in shapes})
        filt = (g_shape_filter_in.value or "").strip().lower()
        names = [n for n in all_names if (filt in n.lower())] if filt else all_names
        items = [(n, n) for n in names] or [(g_shape_dd.value, g_shape_dd.value)]
        cur = g_shape_dd.value
        _set_dd_options_preserve(g_shape_dd, items, prefer=cur or GENERATOR_SHAPE)
        _toggle_gen_flex(g_shape_dd.value)
        g_shape_loading.value = "<span class='nb3-small'></span>"

        # Generator images: list in root, then validate shape compatibility
        g_img_loading.value = "<span class='nb3-small'>loading images…</span>"
        imgs = []
        try:
            resp = oci.pagination.list_call_get_all_results(
                cc.list_images, TENANCY_OCID,
                operating_system="Oracle Linux", sort_by="TIMECREATED", sort_order="DESC"
            )
            imgs = list(resp.data)[:2000]
        except Exception:
            imgs = []
        pairs = []
        used = 0
        for im in imgs:
            if used >= 200:
                break
            try:
                resp = oci.pagination.list_call_get_all_results(
                    cc.list_image_shape_compatibility_entries, image_id=im.id
                )
                shapes = {e.shape for e in (resp.data or [])}
                ok = False
                if g_shape_dd.value in shapes:
                    ok = True
                elif g_shape_dd.value.endswith(".Flex") and g_shape_dd.value[:-5] in shapes:
                    ok = True
                if not ok:
                    continue
                try:
                    created = im.time_created.strftime("%Y-%m-%d")
                except Exception:
                    created = ""
                pairs.append((f"{im.display_name} — {im.operating_system} {im.operating_system_version}" + (f" — {created}" if created else ""), im.id))
                used += 1
            except Exception:
                continue

        _set_dd_options_preserve(g_img_dd, pairs or [("No compatible Oracle Linux images for selected shape", "")], prefer=os.environ.get("GEN_IMAGE_ID","") or (pairs[0][1] if pairs else ""))
        # Preview
        try:
            lab = next((lab for (lab, val) in g_img_dd.options if val == g_img_dd.value), "(none)")
        except Exception:
            lab = "(none)"
        g_img_selected.value = f"<span class='nb3-small'>Selected image: {lab}</span>"
        g_img_loading.value = "<span class='nb3-small'></span>"
    except Exception:
        g_shape_loading.value = "<span class='nb3-small' style='color:#b00020'>shape load error</span>"
        g_img_loading.value   = "<span class='nb3-small' style='color:#b00020'>image load error</span>"

g_shape_filter_in.observe(refresh_gen_shapes_and_images, names="value")
g_shape_dd.observe(lambda ch: (_toggle_gen_flex(g_shape_dd.value), refresh_gen_shapes_and_images()), names="value")
region_dd.observe(refresh_gen_shapes_and_images, names="value")
comp_dd.observe(refresh_gen_shapes_and_images, names="value")
refresh_gen_shapes_and_images()

# ----- Locust & Targets -----
mode_locust_dd  = widgets.Dropdown(options=[("CPS (connections/sec)", "cps"), ("Throughput (GET payload)", "throughput")],
                                   value=os.environ.get("TEST_MODE", "cps"))
wait_in         = widgets.BoundedFloatText(value=float(os.environ.get("LOCUST_WAIT_TIME_SEC","1.0")), min=0.0, max=60.0, step=0.1)
conn_ms_in      = widgets.BoundedIntText(value=int(os.environ.get("LOCUST_CONNECT_TIMEOUT_MS","8000")), min=100, max=60000, step=100)
read_ms_in      = widgets.BoundedIntText(value=int(os.environ.get("LOCUST_READ_TIMEOUT_MS","15000")), min=100, max=120000, step=100)
verify_tls_cb   = widgets.Checkbox(value=os.environ.get("LOCUST_VERIFY_TLS","false").strip().lower() in ("1","true","yes","y"))
ui_port_in      = widgets.BoundedIntText(value=int(os.environ.get("UI_WEB_PORT","8089")), min=1024, max=65535, step=1)
targets_in      = widgets.Textarea(value=os.environ.get("EXTERNAL_TARGETS_TEXT",""), layout=widgets.Layout(height="80px"))
targets_prev    = widgets.HTML("<span class='nb3-small'>Normalized: (none)</span>")
targets_err     = widgets.HTML("")

def _normalize_targets(txt: str) -> list[str]:
    out = []
    for tok in (txt or "").split(","):
        t = (tok or "").strip()
        if not t:
            continue
        if not (t.startswith("http://") or t.startswith("https://")):
            t = "https://" + t
        try:
            scheme, rest = t.split("://", 1)
            host = rest.split("/")[0]
            if ":" in host and not (host.startswith("[") and host.endswith("]")):
                t = f"{scheme}://[{host}]" + rest[len(host):]
        except Exception:
            pass
        if t not in out:
            out.append(t)
    return out

def _update_targets_preview(*_):
    norm = _normalize_targets(targets_in.value or "")
    targets_prev.value = "<span class='nb3-small'>Normalized: " + (", ".join(norm) if norm else "(none)") + "</span>"

targets_in.observe(_update_targets_preview, names="value")
_update_targets_preview()

# ----- Execution & Destroy -----
RUN_INFRA_APPLY       = os.environ.get("RUN_INFRA_APPLY", "true").strip().lower() in ("1","true","yes","y")
RUN_K8S_DEPLOY        = os.environ.get("RUN_K8S_DEPLOY", "true").strip().lower() in ("1","true","yes","y")
RUN_GENERATORS_PREP   = os.environ.get("RUN_GENERATORS_PREP", "true").strip().lower() in ("1","true","yes","y")
RUN_LOCUST            = os.environ.get("RUN_LOCUST", "false").strip().lower() in ("1","true","yes","y")
DESTROY_GENS_ON_APPLY    = os.environ.get("DESTROY_GENS_ON_APPLY", "false").strip().lower() in ("1","true","yes","y")
DESTROY_OKE_ON_APPLY     = os.environ.get("DESTROY_OKE_ON_APPLY",  "false").strip().lower() in ("1","true","yes","y")
DESTROY_NETWORK_ON_APPLY = os.environ.get("DESTROY_NETWORK_ON_APPLY", "false").strip().lower() in ("1","true","yes","y")

run_infra_cb   = widgets.Checkbox(value=RUN_INFRA_APPLY, description="Run: Infra apply")
run_k8s_cb     = widgets.Checkbox(value=RUN_K8S_DEPLOY, description="Run: K8s deploy (TLS+NGINX+LB)")
run_prep_cb    = widgets.Checkbox(value=RUN_GENERATORS_PREP, description="Run: Generators prepare (pip+tmux)")
run_locust_cb  = widgets.Checkbox(value=RUN_LOCUST, description="Run: Locust (master+workers)")
destroy_gens_cb= widgets.Checkbox(value=DESTROY_GENS_ON_APPLY, description="Destroy Generators on apply")
destroy_oke_cb = widgets.Checkbox(value=DESTROY_OKE_ON_APPLY, description="Destroy OKE on apply")
destroy_net_cb = widgets.Checkbox(value=DESTROY_NETWORK_ON_APPLY, description="Destroy Network on apply (only when both disabled)")

# ----- Issues strip, info strip & Apply -----
issues_strip = widgets.HTML("")
pods_info = widgets.HTML(
    "<div class='info-strip'>Direct Pod Routing is fixed in this notebook (pods-as-backends). "
    "The LoadBalancer will target Pod IPs directly and nodePorts will be disabled. "
    "TLS offload + Proxy Protocol v2 remain at the LB. Host networking is not used.</div>"
)
apply_btn = widgets.Button(description="Apply", button_style="primary", icon="check",
                           layout=widgets.Layout(width="240px", height="36px"))
summary_out = widgets.Output()

def _capacity_validate():
    # background capacity validation; only show on error
    w_cidr = (wkp_cidr_in.value or "").strip()
    p_cidr = (pods_cidr_in.value or "").strip()
    try:
        mpp   = max(1, int(max_pods_in.value))
        nodes = max(1, int(pool_size_in.value))
    except Exception:
        return "Invalid max pods/node or node count."
    usable_workers = _usable_ips(w_cidr)
    usable_pods    = _usable_ips(p_cidr)
    if usable_workers < 0 or usable_pods < 0:
        return "Invalid workers_private or pods CIDR."
    sys_p = 64  # hidden system pod budget
    node_cap_by_pods = (usable_pods - sys_p) // mpp if usable_pods > sys_p else 0
    effective_nodes  = min(usable_workers, node_cap_by_pods)
    total_pods_needed = nodes * mpp + sys_p
    if nodes > effective_nodes:
        return "Insufficient node capacity: increase workers_private and/or pods CIDR or reduce Backend nodes."
    if total_pods_needed > usable_pods:
        return "Pods subnet too small: increase pods CIDR or reduce max pods/node or node count."
    return ""

def on_apply_clicked(_b):
    summary_out.clear_output()
    issues_strip.value = ""
    targets_err.value = ""
    node_label_err.value = ""  # reset

    errs = []

    # Generators validation (when enabled)
    gens_enabled = mode_dd.value in ("generators-only", "both")
    if gens_enabled:
        # Require generator image
        if not (g_img_dd.value or "").strip():
            errs.append("Generators image not selected")
        # SSH keys must be present and exist
        if not (ssh_pub_gen_dd.value or "").strip() or not Path(os.path.expanduser(ssh_pub_gen_dd.value)).exists():
            errs.append("GEN SSH public key not found")
        if not (ssh_priv_gen_dd.value or "").strip() or not Path(os.path.expanduser(ssh_priv_gen_dd.value)).exists():
            errs.append("GEN SSH private key not found")
        # Workers policy min/max
        try:
            mn = int(min_workers_in.value)
            mx = int(max_workers_in.value)
            if mn > mx:
                errs.append("Min workers/host cannot exceed Max workers/host")
        except Exception:
            errs.append("Workers/host values must be integers")
        # Targets required in generators-only mode
        if mode_dd.value == "generators-only":
            norm = _normalize_targets(targets_in.value or "")
            if not norm:
                targets_err.value = "<div class='issues-strip'>Targets required for generators-only mode.</div>"
                errs.append("Missing targets")

    # OKE capacity gate (when enabled) — background only
    if mode_dd.value in ("oke-only", "both"):
        cap_msg = _capacity_validate()
        if cap_msg:
            errs.append(cap_msg)
        # Validate node label fields
        lbl_msg = _validate_label_fields(node_label_key_in.value, node_label_val_in.value)
        if lbl_msg:
            node_label_err.value = f"<div class='issues-strip'>{lbl_msg}</div>"
            errs.append("Invalid node pool label key/value")

    if errs:
        issues_strip.value = "<div class='issues-strip'>Apply aborted: " + "; ".join(errs) + "</div>"
        return

    # Persist to env — Location
    os.environ["REGION"]         = region_dd.value
    os.environ["COMPARTMENT_ID"] = comp_dd.value or ""
    os.environ["AD_A"]           = ad_dd.value or ""

    # Mode & Namespace
    os.environ["DEPLOY_MODE"] = mode_dd.value
    os.environ["WORK_NS"]     = ns_in.value.strip() or "cpstest"

    # VCN
    os.environ["CP_CIDR"]              = (cp_cidr_in.value or "").strip()
    os.environ["PUB_LB_CIDR"]          = (plb_cidr_in.value or "").strip()
    os.environ["WORKERS_PRIVATE_CIDR"] = (wkp_cidr_in.value or "").strip()
    os.environ["PODS_CIDR"]            = (pods_cidr_in.value or "").strip()
    os.environ["GENS_PUB_CIDR"]        = (gen_cidr_in.value or "").strip()

    # OKE
    os.environ["KUBERNETES_VERSION"]  = (oke_k8s_dd.value or "").strip()
    os.environ["SSH_PUBLIC_KEY_PATH"] = os.path.expanduser(ssh_pub_oke_dd.value or "")
    os.environ["BACKEND_NODE_SHAPE"]  = (b_shape_dd.value or "").strip()
    os.environ["BACKEND_OCPUS"]       = str(float(b_ocpus_in.value))
    os.environ["BACKEND_MEMORY_GB"]   = str(float(b_mem_in.value))
    os.environ["MAX_PODS_PER_NODE"]   = str(int(max_pods_in.value))
    os.environ["BACKEND_POOL_SIZE"]   = str(int(pool_size_in.value))
    os.environ["NODE_IMAGE_OCID_BACKEND"] = (b_img_dd.value or "").strip()
    os.environ["ORACLE_LINUX_MAJOR"]      = (ol_major_dd.value or "any").replace("OL","").strip().lower()

    # NEW: Persist node pool label key/value
    os.environ["NODE_LABEL_KEY"]   = (node_label_key_in.value or "").strip()
    os.environ["NODE_LABEL_VALUE"] = (node_label_val_in.value or "").strip()

    # Generators
    os.environ["GENERATOR_COUNT"]     = str(int(g_count_in.value))
    os.environ["GENERATOR_SHAPE"]     = (g_shape_dd.value or "").strip()
    os.environ["GENERATOR_OCPUS"]     = str(float(g_ocpus_in.value))
    os.environ["GENERATOR_MEMORY_GB"] = str(float(g_mem_in.value))
    os.environ["ENABLE_IPV6_ON_GENERATORS"] = "true" if bool(g_v6_cb.value) else "false"
    os.environ["GEN_IMAGE_ID"]        = (g_img_dd.value or "").strip()
    os.environ["GEN_SSH_PUBLIC_KEY_PATH"]  = os.path.expanduser(ssh_pub_gen_dd.value or "")
    os.environ["GEN_SSH_PRIVATE_KEY_PATH"] = os.path.expanduser(ssh_priv_gen_dd.value or "")
    # Also set generic SSH_* for downstream self-contained steps
    os.environ["SSH_PUBLIC_KEY_PATH"]  = os.environ["GEN_SSH_PUBLIC_KEY_PATH"]
    os.environ["SSH_PRIVATE_KEY_PATH"] = os.environ["GEN_SSH_PRIVATE_KEY_PATH"]

    # Workers/host CPU policy (canonical only)
    mode = (workers_per_host_mode_in.value or "auto").strip().lower()
    if mode == "auto":
        os.environ["WORKERS_PER_HOST"] = "auto"
    else:
        os.environ["WORKERS_PER_HOST"] = str(int(fixed_workers_in.value))
    os.environ["CPU_RESERVE"]          = str(int(cpu_reserve_in.value))
    os.environ["MIN_WORKERS_PER_HOST"] = str(int(min_workers_in.value))
    os.environ["MAX_WORKERS_PER_HOST"] = str(int(max_workers_in.value))

    # Locust & Targets
    os.environ["TEST_MODE"]                 = (mode_locust_dd.value or "cps").strip().lower()
    os.environ["LOCUST_WAIT_TIME_SEC"]      = str(float(wait_in.value))
    os.environ["LOCUST_CONNECT_TIMEOUT_MS"] = str(int(conn_ms_in.value))
    os.environ["LOCUST_READ_TIMEOUT_MS"]    = str(int(read_ms_in.value))
    os.environ["LOCUST_VERIFY_TLS"]         = "true" if bool(verify_tls_cb.value) else "false"
    os.environ["UI_WEB_PORT"]               = str(int(ui_port_in.value))
    os.environ["EXTERNAL_TARGETS_TEXT"]     = ",".join(_normalize_targets(targets_in.value or ""))

    # Paths/endpoints (persist from SSOT; no UI in this cell)
    os.environ["LOCUST_WORKDIR"]           = os.environ.get("LOCUST_WORKDIR", "/home/opc/locustwork")
    os.environ["HEALTH_ENDPOINT_PATH"]     = os.environ.get("HEALTH_ENDPOINT_PATH", "/healthz")
    os.environ["THROUGHPUT_ENDPOINT_PATH"] = os.environ.get("THROUGHPUT_ENDPOINT_PATH", "/payload_100k")

    # Toggles
    os.environ["RUN_INFRA_APPLY"]     = "true"  if bool(run_infra_cb.value) else "false"
    os.environ["RUN_K8S_DEPLOY"]      = "true"  if bool(run_k8s_cb.value)   else "false"
    os.environ["RUN_GENERATORS_PREP"] = "true"  if bool(run_prep_cb.value)  else "false"
    os.environ["RUN_LOCUST"]          = "true"  if bool(run_locust_cb.value)else "false"
    os.environ["DESTROY_GENS_ON_APPLY"]    = "true" if bool(destroy_gens_cb.value) else "false"
    os.environ["DESTROY_OKE_ON_APPLY"]     = "true" if bool(destroy_oke_cb.value)  else "false"
    os.environ["DESTROY_NETWORK_ON_APPLY"] = "true" if bool(destroy_net_cb.value)  else "false"

    # Persist the fixed pods-as-backends choice for downstream cells (informational)
    os.environ["SERVICE_BACKEND_MODE"] = "pods"
    os.environ["POD_BACKEND_PORT"]     = os.environ.get("POD_BACKEND_PORT", "30080")

    # Summary
    lines = []
    lines.append(f"region={os.environ['REGION']}  compartment={os.environ['COMPARTMENT_ID']}  ad={os.environ['AD_A']}")
    lines.append(f"mode={os.environ['DEPLOY_MODE']}  ns={os.environ['WORK_NS']}")
    lines.append(f"VCN: cp={os.environ['CP_CIDR']} pub_lb={os.environ['PUB_LB_CIDR']} workers_private={os.environ['WORKERS_PRIVATE_CIDR']} pods={os.environ['PODS_CIDR']} gens_pub={os.environ['GENS_PUB_CIDR']}")
    lines.append(f"OKE: k8s={os.environ['KUBERNETES_VERSION']} shape={os.environ['BACKEND_NODE_SHAPE']} ocpus={os.environ['BACKEND_OCPUS']} memGB={os.environ['BACKEND_MEMORY_GB']} max_pods/node={os.environ['MAX_PODS_PER_NODE']} backend_nodes={os.environ['BACKEND_POOL_SIZE']}")
    lines.append(f"Worker image override={(os.environ['NODE_IMAGE_OCID_BACKEND'] or '(default)')}, OL_major={os.environ['ORACLE_LINUX_MAJOR']}")
    # NEW: node pool label
    lines.append(f"Node pool label: {os.environ['NODE_LABEL_KEY']}={os.environ['NODE_LABEL_VALUE']}")
    lines.append(f"GEN: count={os.environ['GENERATOR_COUNT']} shape={os.environ['GENERATOR_SHAPE']} ocpus={os.environ['GENERATOR_OCPUS']} memGB={os.environ['GENERATOR_MEMORY_GB']} v6={os.environ['ENABLE_IPV6_ON_GENERATORS']}")
    # Workers policy summary
    wph = os.environ.get("WORKERS_PER_HOST","auto")
    if wph == "auto":
        lines.append(f"Workers policy: mode=auto | reserve={os.environ.get('CPU_RESERVE')} | min={os.environ.get('MIN_WORKERS_PER_HOST')} | max={os.environ.get('MAX_WORKERS_PER_HOST')}")
    else:
        lines.append(f"Workers policy: mode=fixed | fixed={wph}")
    lines.append(f"Locust: mode={os.environ['TEST_MODE']} wait(s)={os.environ['LOCUST_WAIT_TIME_SEC']} connect(ms)={os.environ['LOCUST_CONNECT_TIMEOUT_MS']} read(ms)={os.environ['LOCUST_READ_TIMEOUT_MS']} verify_tls={os.environ['LOCUST_VERIFY_TLS']} ui_port={os.environ['UI_WEB_PORT']}")
    lines.append(f"targets={(os.environ['EXTERNAL_TARGETS_TEXT'] or '(none)')}")
    lines.append(f"pods-as-backends: backend_mode={os.environ.get('SERVICE_BACKEND_MODE','pods')} pod_port={os.environ.get('POD_BACKEND_PORT','30080')}")
    lines.append(f"toggles: run_infra={os.environ['RUN_INFRA_APPLY']} run_k8s={os.environ['RUN_K8S_DEPLOY']} run_prep={os.environ['RUN_GENERATORS_PREP']} run_locust={os.environ['RUN_LOCUST']} destroy_gens={os.environ['DESTROY_GENS_ON_APPLY']} destroy_oke={os.environ['DESTROY_OKE_ON_APPLY']} destroy_net={os.environ['DESTROY_NETWORK_ON_APPLY']}")

    summary_out.clear_output()
    with summary_out:
        display(widgets.HTML(value="<b>Selections applied</b>"))
        display(widgets.HTML(value=f"<div class='nb3-pre'>{'<br/>'.join(lines)}</div>"))
    print("Cell 3 complete: Selections applied.")

apply_btn.on_click(on_apply_clicked)

# ----- Compose sections (no Capacity Estimator UI; validation runs on Apply only) -----
sec_location = section("Location",
    loc_error,
    row("Region:", region_dd), widgets.HBox([reload_btn, reset_btn], layout=widgets.Layout(gap="8px")),
    row("Compartment:", comp_dd),
    row("Availability Domain:", ad_dd)
)

sec_mode = section("Mode & Namespace",
    row("Deploy mode:", mode_dd),
    row("K8s namespace:", ns_in)
)

sec_vcn  = section("One‑VCN Networking",
    row("cp CIDR:",   cp_cidr_in),
    row("pub_lb CIDR:", plb_cidr_in),
    row("workers_private CIDR:", wkp_cidr_in),
    row("pods CIDR:", pods_cidr_in),
    row("gens_pub CIDR:", gen_cidr_in)
)

sec_oke  = section("OKE (Backends only)",
    row("OKE K8s version:", oke_k8s_dd), widgets.HBox([oke_k8s_reload, oke_versions_info], layout=widgets.Layout(gap="8px")),
    row("Shape filter:", shape_filter_in), shape_loading_lbl,
    row("Backend shape:", b_shape_dd),
    row("Backend OCPUs:", b_ocpus_in),
    row("Backend Memory (GB):", b_mem_in),
    row("Oracle Linux major:", ol_major_dd),
    row("Worker image (override):", b_img_dd), b_img_selected, b_img_note, b_img_diag,
    row("SSH pub (OKE):", ssh_pub_oke_dd),
    row("Max pods/node:", max_pods_in),
    row("Backend nodes:", pool_size_in),
    # NEW rows: node pool label key/value + validation hint
    row("Node pool label key:", node_label_key_in),
    row("Node pool label value:", node_label_val_in),
    node_label_err
)

sec_gen  = section("Generators (Compute)",
    row("Shape filter:", g_shape_filter_in), g_shape_loading,
    row("Generator shape:", g_shape_dd),
    row("Generator OCPUs:", g_ocpus_in),
    row("Generator Memory (GB):", g_mem_in),
    row("Generator image:", g_img_dd), g_img_selected, g_img_loading,
    row("SSH pub (GEN):", ssh_pub_gen_dd),
    row("SSH priv (GEN):", ssh_priv_gen_dd),
    row("Workers/host mode:", workers_per_host_mode_in),
    row("Fixed workers/host:", fixed_workers_in),
    row("CPU reserve (auto):", cpu_reserve_in),
    row("Min workers/host:", min_workers_in),
    row("Max workers/host:", max_workers_in),
    lbl_err_workers,
    row("Generators count:", g_count_in),
    row("Assign IPv6 to generators:", g_v6_cb)
)

sec_loc  = section("Locust & Targets",
    row("Locust mode:", mode_locust_dd),
    row("Wait(s)/user:", wait_in),
    row("Connect ms:",   conn_ms_in),
    row("Read ms:",      read_ms_in),
    row("Verify TLS:",   verify_tls_cb),
    row("UI port:",      ui_port_in),
    row("External targets:", targets_in), targets_prev, targets_err
)

# Mode-based visibility
def _toggle_by_mode(*_):
    is_oke = (mode_dd.value in ("oke-only", "both"))
    sec_oke.layout.display = "" if is_oke else "none"
    is_gen = (mode_dd.value in ("generators-only", "both"))
    sec_gen.layout.display = "" if is_gen else "none"

_toggle_by_mode()
mode_dd.observe(_toggle_by_mode, names="value")

container = widgets.VBox(
    [pods_info,
     issues_strip,
     sec_location, sec_mode, sec_vcn, sec_oke, sec_gen, sec_loc,
     section("Execution & Destroy",
         run_infra_cb, run_k8s_cb, run_prep_cb, run_locust_cb,
         destroy_gens_cb, destroy_oke_cb, destroy_net_cb
     ),
     widgets.HBox([apply_btn], layout=widgets.Layout(justify_content="center")),
     summary_out],
    layout=widgets.Layout(width="100%", align_items="stretch", justify_content="flex-start"),
    _dom_classes=['nb3-container']
)

display(container)
print("Cell 3 loaded: Region-scoped OKE images (CE sources), version/shape/arch/OL-major filtered. "
      "Direct Pod Routing is fixed; Service will use pods as backends (LB→Pod IPs). Configure, then click Apply.")


In [ ]:
# Cell 4 — Objective: Generate combined Terraform stack (One‑VCN + conditional OKE backend + conditional Generators) and cloud‑init
# - Writes ./stack with: provider.tf, variables.tf, locals.tf, network.tf, oke.tf, generators.tf, outputs.tf, terraform.tfvars
# - Writes ./stack/cloud-init/generator.sh (tuning-only; no package installs) including a multi-capable locustfile.py
# - Honors DEPLOY_MODE from Cell 3 → enable_oke/enable_generators
# - Uses ONLY centralized values from Cell 2/3 (read from os.environ). No hardcoded secrets or paths here.
# - Security model: FULL STATELESS posture — Default Security List and all NSGs have symmetric stateless rules for IPv4 and IPv6
#   (protocol="all", stateless=true) in both directions, ensuring explicit return paths for stateless flows.
# - UPDATE: Export NSG OCIDs (nsg_lb_id, nsg_backends_id, nsg_generators_id) so Cell 7 can consume them deterministically.
# - NEW: Wire node pool initial labels (node_label_key/node_label_value) so all nodes in the backend pool are labeled on creation (top-level block, not inside node_config_details).

import os
from pathlib import Path
import textwrap

def _fail(msg: str):
    raise ValueError(msg)

# Read authoritative config from environment (set by Cell 2 and persisted by Cell 3 Apply)
OCI_PROFILE          = os.environ.get("OCI_PROFILE", "DEFAULT").strip()
REGION               = os.environ.get("REGION", "").strip()
COMPARTMENT_ID       = os.environ.get("COMPARTMENT_ID", "").strip()
AD_A                 = os.environ.get("AD_A", "").strip()

DEPLOY_MODE          = os.environ.get("DEPLOY_MODE", "both").strip().lower()
WORK_NS              = os.environ.get("WORK_NS", "lbtest").strip()

# One‑VCN CIDRs
CP_CIDR              = os.environ.get("CP_CIDR", "10.0.1.0/24").strip()
PUB_LB_CIDR          = os.environ.get("PUB_LB_CIDR", "10.0.2.0/24").strip()
WORKERS_PRIVATE_CIDR = os.environ.get("WORKERS_PRIVATE_CIDR", "10.0.4.0/22").strip()
PODS_CIDR            = os.environ.get("PODS_CIDR", "10.0.64.0/18").strip()
GENS_PUB_CIDR        = os.environ.get("GENS_PUB_CIDR", "10.0.20.0/24").strip()

# OKE (Backends only)
KUBERNETES_VERSION      = os.environ.get("KUBERNETES_VERSION", "v1.34.2").strip()
SSH_PUBLIC_KEY_PATH_OKE = os.path.expanduser(os.environ.get("SSH_PUBLIC_KEY_PATH", ""))  # used by node pool
BACKEND_NODE_SHAPE      = os.environ.get("BACKEND_NODE_SHAPE", "VM.Standard.E5.Flex").strip()
BACKEND_OCPUS           = os.environ.get("BACKEND_OCPUS", "16").strip()
BACKEND_MEMORY_GB       = os.environ.get("BACKEND_MEMORY_GB", "64").strip()
MAX_PODS_PER_NODE       = os.environ.get("MAX_PODS_PER_NODE", "31").strip()
BACKEND_POOL_SIZE       = os.environ.get("BACKEND_POOL_SIZE", "1").strip()
NODE_IMAGE_OCID_BACKEND = os.environ.get("NODE_IMAGE_OCID_BACKEND", "").strip()  # required for OKE in this stack

# NEW — Node pool initial labels for backend scheduling (SSOT from Cells 2/3)
NODE_LABEL_KEY          = os.environ.get("NODE_LABEL_KEY", "app.role").strip()
NODE_LABEL_VALUE        = os.environ.get("NODE_LABEL_VALUE", "backend").strip()

# Generators (Compute)
GENERATOR_COUNT      = os.environ.get("GENERATOR_COUNT", "1").strip()
GENERATOR_SHAPE      = os.environ.get("GENERATOR_SHAPE", "VM.Standard.E5.Flex").strip()
GENERATOR_OCPUS      = os.environ.get("GENERATOR_OCPUS", "8").strip()
GENERATOR_MEMORY_GB  = os.environ.get("GENERATOR_MEMORY_GB", "32").strip()
ENABLE_IPV6_ON_GENERATORS = os.environ.get("ENABLE_IPV6_ON_GENERATORS", "false").strip().lower() in ("1","true","yes","y")
GEN_IMAGE_ID         = os.path.expanduser(os.environ.get("GEN_IMAGE_ID", "")).strip()  # generator image (Oracle Linux) chosen in Cell 3

# SSH keys for Generators (authorized_keys content + path to private key for later SSH-based steps)
GEN_SSH_PUBLIC_KEY_PATH  = os.path.expanduser(os.environ.get("GEN_SSH_PUBLIC_KEY_PATH", "")).strip()
GEN_SSH_PRIVATE_KEY_PATH = os.path.expanduser(os.environ.get("GEN_SSH_PRIVATE_KEY_PATH", "")).strip()

# Compute enable flags from DEPLOY_MODE (authoritative)
ENABLE_OKE        = DEPLOY_MODE in ("oke-only", "both")
ENABLE_GENERATORS = DEPLOY_MODE in ("generators-only", "both")

# Basic validations (fail-fast; NO secrets printed)
if not REGION:              _fail("REGION is empty. Ensure Cell 3 Apply persisted it.")
if not COMPARTMENT_ID:      _fail("COMPARTMENT_ID is empty. Ensure Cell 3 Apply persisted it.")
if not AD_A:                _fail("AD_A is empty. Ensure Cell 3 Apply persisted it.")
if ENABLE_OKE and (not SSH_PUBLIC_KEY_PATH_OKE or not Path(SSH_PUBLIC_KEY_PATH_OKE).exists()):
    _fail(f"OKE SSH public key not found. Current: {SSH_PUBLIC_KEY_PATH_OKE or '(unset)'}")
if ENABLE_OKE and not NODE_IMAGE_OCID_BACKEND:
    _fail("NODE_IMAGE_OCID_BACKEND is empty. Provide an OKE worker image override (Cell 3).")
if ENABLE_GENERATORS:
    if not GEN_SSH_PUBLIC_KEY_PATH or not Path(GEN_SSH_PUBLIC_KEY_PATH).exists():
        _fail(f"GEN SSH public key not found. Current: {GEN_SSH_PUBLIC_KEY_PATH or '(unset)'}")
    if not GEN_SSH_PRIVATE_KEY_PATH or not Path(GEN_SSH_PRIVATE_KEY_PATH).exists():
        _fail(f"GEN SSH private key not found. Current: {GEN_SSH_PRIVATE_KEY_PATH or '(unset)'}")
    with open(GEN_SSH_PUBLIC_KEY_PATH, "r") as f:
        GEN_SSH_PUBLIC_KEY_CONTENT = (f.read() or "").strip()
    if not GEN_SSH_PUBLIC_KEY_CONTENT:
        _fail("GEN SSH public key file is empty.")
    if not GEN_IMAGE_ID:
        _fail("GEN_IMAGE_ID (generator image OCID) is empty. Select a generator image in Cell 3.")
else:
    GEN_SSH_PUBLIC_KEY_CONTENT = ""  # unused in this mode

# Prepare directories
stack_dir = Path("stack")
cloud_init_dir = stack_dir / "cloud-init"
stack_dir.mkdir(parents=True, exist_ok=True)
cloud_init_dir.mkdir(parents=True, exist_ok=True)

# --------- Terraform files content ---------

provider_tf = textwrap.dedent(f"""
terraform {{
  required_providers {{
    oci = {{
      source  = "oracle/oci"
      version = ">= 7.30.0"
    }}
    random = {{
      source  = "hashicorp/random"
      version = ">= 3.4.3"
    }}
  }}
}}

provider "oci" {{
  config_file_profile = var.oci_profile
  region              = var.region
}}
""").strip("\n")

variables_tf = textwrap.dedent("""
# Global/auth
variable "oci_profile" { type = string }
variable "region"      { type = string }
variable "compartment_id" { type = string }
variable "ad_a"        { type = string }

# Feature toggles
variable "enable_oke"        { type = bool }
variable "enable_generators" { type = bool }

# One‑VCN CIDRs
variable "cp_cidr"              { type = string }
variable "pub_lb_cidr"          { type = string }
variable "workers_private_cidr" { type = string }
variable "pods_cidr"            { type = string }
variable "gens_pub_cidr"        { type = string }

# OKE (Backends only)
variable "kubernetes_version"      { type = string }
variable "ssh_public_key_path_oke" { type = string }
variable "backend_node_shape"      { type = string }
variable "backend_ocpus"           { type = number }
variable "backend_memory_gb"       { type = number }
variable "max_pods_per_node"       { type = number }
variable "backend_pool_size"       { type = number }
variable "node_image_ocid_backend" { type = string }

# NEW — Node pool initial labels (applied at creation on backend node pool)
variable "node_label_key"   { type = string }
variable "node_label_value" { type = string }

# Generators (Compute)
variable "generator_count"        { type = number }
variable "generator_shape"        { type = string }
variable "generator_ocpus"        { type = number }
variable "generator_memory_gb"    { type = number }
variable "enable_ipv6_on_generators" { type = bool }
variable "generator_image_id"     { type = string }
variable "ssh_public_key_content_gen" { type = string }
variable "ssh_private_key_path_gen"   { type = string }

# Misc
variable "bastion_plugin_name" {
  type    = string
  default = "Bastion"
}
""").strip("\n")

locals_tf = textwrap.dedent("""
resource "random_string" "state" {
  length  = 6
  upper   = false
  numeric = false
  special = false
}

locals {
  # Name prefix for resources (short, stable)
  name          = "cps-${random_string.state.result}"
  vcn_dns_label = "cps${random_string.state.result}"
}
""").strip("\n")

# Network + NSGs (TEST MODEL: COMPLETE STATELESS posture)
network_tf = textwrap.dedent("""
# network.tf — TEST MODEL (stateless “all/all”) with explicit return path rules
# - Default Security List: protocol="all" stateless egress+ingress (v4+v6) for complete stateless posture.
# - NSGs (lb/backends/generators): symmetric stateless egress+ingress (v4+v6) to avoid constraining return paths.
# - Private RT: NAT for IPv4 only (no ::/0), public RT: v4+v6 via IGW.

# VCN dual-stack (test harness)
resource "oci_core_vcn" "vcn" {
  cidr_block     = "10.0.0.0/16"  # supernet anchor (not used for routing directly)
  compartment_id = var.compartment_id
  display_name   = local.name
  is_ipv6enabled = true
  dns_label      = local.vcn_dns_label
}

# Route tables
resource "oci_core_internet_gateway" "igw" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "${local.name}-igw"
}

resource "oci_core_nat_gateway" "nat" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "${local.name}-nat"
}

# Public RT (IPv4+IPv6 to IGW)
resource "oci_core_route_table" "rt_public" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "${local.name}-rt-public"

  route_rules {
    destination       = "0.0.0.0/0"
    destination_type  = "CIDR_BLOCK"
    network_entity_id = oci_core_internet_gateway.igw.id
  }

  route_rules {
    destination       = "::/0"
    destination_type  = "CIDR_BLOCK"
    network_entity_id = oci_core_internet_gateway.igw.id
  }
}

# Private RT (IPv4 to NAT only; NO ::/0)
resource "oci_core_route_table" "rt_private" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "${local.name}-rt-private"

  route_rules {
    destination       = "0.0.0.0/0"
    destination_type  = "CIDR_BLOCK"
    network_entity_id = oci_core_nat_gateway.nat.id
  }
}

# Default Security List — TEST MODEL (NOT PROD): protocol "all" stateless egress+ingress (v4+v6)
resource "oci_core_default_security_list" "vcn_default" {
  manage_default_resource_id = oci_core_vcn.vcn.default_security_list_id

  # Egress all v4
  egress_security_rules {
    protocol         = "all"
    destination      = "0.0.0.0/0"
    destination_type = "CIDR_BLOCK"
    stateless        = true
  }

  # Egress all v6
  egress_security_rules {
    protocol         = "all"
    destination      = "::/0"
    destination_type = "CIDR_BLOCK"
    stateless        = true
  }

  # Ingress all v4
  ingress_security_rules {
    protocol    = "all"
    source      = "0.0.0.0/0"
    source_type = "CIDR_BLOCK"
    stateless   = true
  }

  # Ingress all v6
  ingress_security_rules {
    protocol    = "all"
    source      = "::/0"
    source_type = "CIDR_BLOCK"
    stateless   = true
  }
}

# Derive IPv6 subnets from the VCN base
locals {
  vcn_ipv6_base   = tolist(oci_core_vcn.vcn.ipv6cidr_blocks)[0]
  cp_ipv6_cidr    = cidrsubnet(local.vcn_ipv6_base, 8, 1)
  plb_ipv6_cidr   = cidrsubnet(local.vcn_ipv6_base, 8, 2)
  wprv_ipv6_cidr  = cidrsubnet(local.vcn_ipv6_base, 8, 3)
  pods_ipv6_cidr  = cidrsubnet(local.vcn_ipv6_base, 8, 4)
  gens_ipv6_cidr  = cidrsubnet(local.vcn_ipv6_base, 8, 5)
}

# Subnets
resource "oci_core_subnet" "cp" {
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  display_name               = "${local.name}-cp"
  cidr_block                 = var.cp_cidr
  ipv6cidr_block             = local.cp_ipv6_cidr
  prohibit_public_ip_on_vnic = false
  route_table_id             = oci_core_route_table.rt_public.id
  dns_label                  = "cp"
}

resource "oci_core_subnet" "pub_lb" {
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  display_name               = "${local.name}-pub-lb"
  cidr_block                 = var.pub_lb_cidr
  ipv6cidr_block             = local.plb_ipv6_cidr
  prohibit_public_ip_on_vnic = false
  route_table_id             = oci_core_route_table.rt_public.id
  dns_label                  = "publb"
}

resource "oci_core_subnet" "workers_private" {
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  display_name               = "${local.name}-workers-prv"
  cidr_block                 = var.workers_private_cidr
  ipv6cidr_block             = local.wprv_ipv6_cidr
  prohibit_public_ip_on_vnic = true
  route_table_id             = oci_core_route_table.rt_private.id
  dns_label                  = "wksprv"
}

resource "oci_core_subnet" "pods" {
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  display_name               = "${local.name}-pods"
  cidr_block                 = var.pods_cidr
  ipv6cidr_block             = local.pods_ipv6_cidr
  prohibit_public_ip_on_vnic = true
  route_table_id             = oci_core_route_table.rt_private.id
  dns_label                  = "pods"
}

resource "oci_core_subnet" "gens_pub" {
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  display_name               = "${local.name}-gens-pub"
  cidr_block                 = var.gens_pub_cidr
  ipv6cidr_block             = local.gens_ipv6_cidr
  prohibit_public_ip_on_vnic = false
  route_table_id             = oci_core_route_table.rt_public.id
  dns_label                  = "genspub"
}

# NSGs — TEST MODEL (NOT PROD): "all/all stateless" in both directions and both families

resource "oci_core_network_security_group" "nsg_lb" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "${local.name}-nsg-lb"
}

resource "oci_core_network_security_group_security_rule" "nsg_lb_egress_v4" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "0.0.0.0/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "nsg_lb_egress_v6" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "::/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "nsg_lb_ingress_v4" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "INGRESS"
  protocol                  = "all"
  source_type               = "CIDR_BLOCK"
  source                    = "0.0.0.0/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "nsg_lb_ingress_v6" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "INGRESS"
  protocol                  = "all"
  source_type               = "CIDR_BLOCK"
  source                    = "::/0"
  stateless                 = true
}

resource "oci_core_network_security_group" "nsg_backends" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "${local.name}-nsg-backends"
}

resource "oci_core_network_security_group_security_rule" "nsg_back_egress_v4" {
  network_security_group_id = oci_core_network_security_group.nsg_backends.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "0.0.0.0/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "nsg_back_egress_v6" {
  network_security_group_id = oci_core_network_security_group.nsg_backends.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "::/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "nsg_back_ingress_v4" {
  network_security_group_id = oci_core_network_security_group.nsg_backends.id
  direction                 = "INGRESS"
  protocol                  = "all"
  source_type               = "CIDR_BLOCK"
  source                    = "0.0.0.0/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "nsg_back_ingress_v6" {
  network_security_group_id = oci_core_network_security_group.nsg_backends.id
  direction                 = "INGRESS"
  protocol                  = "all"
  source_type               = "CIDR_BLOCK"
  source                    = "::/0"
  stateless                 = true
}

resource "oci_core_network_security_group" "nsg_generators" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "${local.name}-nsg-generators"
}

resource "oci_core_network_security_group_security_rule" "gens_egress_v4" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "0.0.0.0/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "gens_egress_v6" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "::/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "gens_ingress_v4" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "all"
  source_type               = "CIDR_BLOCK"
  source                    = "0.0.0.0/0"
  stateless                 = true
}

resource "oci_core_network_security_group_security_rule" "gens_ingress_v6" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "all"
  source_type               = "CIDR_BLOCK"
  source                    = "::/0"
  stateless                 = true
}
""").strip("\n")

# FIXED oke.tf — initial_node_labels at TOP-LEVEL in node pool (not inside node_config_details)
oke_tf = textwrap.dedent("""
# OKE cluster and backend node pool (conditional via count)
resource "oci_containerengine_cluster" "this" {
  count              = var.enable_oke ? 1 : 0
  compartment_id     = var.compartment_id
  name               = local.name
  kubernetes_version = var.kubernetes_version
  vcn_id             = oci_core_vcn.vcn.id

  # Native CNI (TOP-LEVEL)
  cluster_pod_network_options {
    cni_type = "OCI_VCN_IP_NATIVE"
  }

  endpoint_config {
    is_public_ip_enabled = true
    subnet_id            = oci_core_subnet.cp.id
  }

  options {
    # service LB on public subnet; IPv4 only for stability
    service_lb_subnet_ids = [ oci_core_subnet.pub_lb.id ]
    ip_families = ["IPv4"]
  }
}

resource "oci_containerengine_node_pool" "backend" {
  count              = var.enable_oke ? 1 : 0
  compartment_id     = var.compartment_id
  cluster_id         = oci_containerengine_cluster.this[0].id
  kubernetes_version = var.kubernetes_version
  name               = "${local.name}-backend"
  node_shape         = var.backend_node_shape
  ssh_public_key     = file(var.ssh_public_key_path_oke)

  # Correct placement for initial_node_labels (top-level)
  initial_node_labels {
    key   = var.node_label_key
    value = var.node_label_value
  }

  node_config_details {
    size = var.backend_pool_size

    placement_configs {
      availability_domain = var.ad_a
      subnet_id           = oci_core_subnet.workers_private.id
    }

    node_pool_pod_network_option_details {
      cni_type          = "OCI_VCN_IP_NATIVE"
      max_pods_per_node = var.max_pods_per_node
      pod_nsg_ids       = []
      pod_subnet_ids    = [ oci_core_subnet.pods.id ]
    }
  }

  # Flex shape config if applicable (safe across fixed shapes; ignored when not Flex)
  node_shape_config {
    ocpus         = var.backend_ocpus
    memory_in_gbs = var.backend_memory_gb
  }

  node_source_details {
    source_type = "IMAGE"
    image_id    = var.node_image_ocid_backend
  }
}
""").strip("\n")

generators_tf = textwrap.dedent("""
# Generator instances (conditional by enable_generators)
resource "oci_core_instance" "generator" {
  count               = var.enable_generators ? var.generator_count : 0
  availability_domain = var.ad_a
  compartment_id      = var.compartment_id
  shape               = var.generator_shape

  # Flex shape config if applicable
  dynamic "shape_config" {
    for_each = endswith(var.generator_shape, ".Flex") ? [1] : []
    content {
      ocpus         = var.generator_ocpus
      memory_in_gbs = var.generator_memory_gb
    }
  }

  source_details {
    source_type = "image"
    source_id   = var.generator_image_id
  }

  create_vnic_details {
    subnet_id        = oci_core_subnet.gens_pub.id
    assign_public_ip = true
    nsg_ids          = [ oci_core_network_security_group.nsg_generators.id ]
  }

  agent_config {
    are_all_plugins_disabled = false
    is_management_disabled   = false
    is_monitoring_disabled   = false

    plugins_config {
      name          = var.bastion_plugin_name
      desired_state = "ENABLED"
    }
  }

  display_name = "ext-gen-${count.index}"

  metadata = {
    ssh_authorized_keys = var.ssh_public_key_content_gen
    user_data           = filebase64("${path.module}/cloud-init/generator.sh")
  }
}

# Optional IPv6 assignment per generator (only if enabled)
data "oci_core_vnic_attachments" "gen_vnic" {
  count          = (var.enable_generators && var.enable_ipv6_on_generators) ? var.generator_count : 0
  compartment_id = var.compartment_id
  instance_id    = oci_core_instance.generator[count.index].id
}

resource "oci_core_ipv6" "gen_v6" {
  count     = (var.enable_generators && var.enable_ipv6_on_generators) ? var.generator_count : 0
  subnet_id = oci_core_subnet.gens_pub.id
  vnic_id   = data.oci_core_vnic_attachments.gen_vnic[count.index].vnic_attachments[0].vnic_id
}
""").strip("\n")

# UPDATED outputs: include NSG IDs so Cell 7 can consume deterministically
outputs_tf = textwrap.dedent("""
output "vcn_id" {
  value = oci_core_vcn.vcn.id
}

output "subnets" {
  value = {
    control_plane   = oci_core_subnet.cp.id
    public_lb       = oci_core_subnet.pub_lb.id
    workers_private = oci_core_subnet.workers_private.id
    pods            = oci_core_subnet.pods.id
    gens_pub        = oci_core_subnet.gens_pub.id
  }
}

output "cluster_id" {
  value = var.enable_oke ? oci_containerengine_cluster.this[0].id : null
}

output "gen_public_ips" {
  value = var.enable_generators ? [for i in oci_core_instance.generator : i.public_ip] : []
}

output "gen_ipv6_ips" {
  value = var.enable_generators ? oci_core_ipv6.gen_v6[*].ip_address : []
}

# NEW — Export NSG OCIDs so Cell 7 can read from stack-outputs.json
output "nsg_lb_id" {
  value = oci_core_network_security_group.nsg_lb.id
}

output "nsg_backends_id" {
  value = oci_core_network_security_group.nsg_backends.id
}

output "nsg_generators_id" {
  value = oci_core_network_security_group.nsg_generators.id
}
""").strip("\n")

# --------- cloud-init: generator.sh (tuning-only; includes locustfile.py) ---------
generator_cloud_init = r"""#!/bin/bash
set -euo pipefail

# Stop/disable firewalld if present
if command -v firewall-cmd >/dev/null 2>&1; then
  systemctl stop firewalld || true
  systemctl disable firewalld || true
fi

# Enable Oracle Cloud Agent if present
if systemctl list-unit-files | grep -q oracle-cloud-agent.service; then
  systemctl enable --now oracle-cloud-agent || true
fi

# Kernel tuning (safe subset)
cat <<'EOF' >/etc/sysctl.d/99-cps.conf
net.core.somaxconn=65535
net.core.netdev_max_backlog=250000
net.ipv4.tcp_max_syn_backlog=262144
net.ipv4.ip_local_port_range=1024 65535
net.ipv4.tcp_fin_timeout=15
fs.file-max=1000000
EOF
sysctl --system || true

# Raise ulimit
echo "* - nofile 1048576" >> /etc/security/limits.conf || true

# Workspace
mkdir -p /home/opc/locustwork/results
chown -R opc:opc /home/opc/locustwork

# Multi-capable locustfile (targets & behavior driven by environment)
cat > /home/opc/locustwork/locustfile.py <<'PY'
import os
from itertools import cycle
from locust import HttpUser, task, constant

VERIFY_TLS        = os.environ.get("LOCUST_VERIFY_TLS", "false").lower() == "true"
CONNECT_TIMEOUT_S = float(os.environ.get("LOCUST_CONNECT_TIMEOUT_S", "8"))
READ_TIMEOUT_S    = float(os.environ.get("LOCUST_READ_TIMEOUT_S", "15"))
WAIT_TIME_S       = float(os.environ.get("LOCUST_WAIT_TIME_S", "1.0"))
MODE              = os.environ.get("LOCUST_MODE", "cps").lower()
HEALTH_PATH       = os.environ.get("LOCUST_HEALTH_PATH", "/healthz")
THROUGHPUT_PATH   = os.environ.get("LOCUST_THROUGHPUT_PATH", "/payload_100k")
RAW_TARGETS       = os.environ.get("LOCUST_TARGETS", "")
TARGETS           = [t.strip() for t in RAW_TARGETS.split(",") if t.strip()]
NAME_BY_VIP       = os.environ.get("LOCUST_NAME_BY_VIP", "true").lower() == "true"

class CpsUser(HttpUser):
    host = os.environ.get("LOCUST_DEFAULT_HOST", "")
    wait_time = constant(WAIT_TIME_S)

    def on_start(self):
        self._targets = cycle(TARGETS) if TARGETS else None

    def _next_base(self):
        try:
            return next(self._targets) if self._targets else (self.host or "")
        except Exception:
            return self.host or ""

    @task
    def do_request(self):
        path = HEALTH_PATH if MODE == "cps" else THROUGHPUT_PATH
        base = self._next_base()
        url  = f"{base}{path}" if base else path
        headers = {"Connection": "close"} if MODE == "cps" else {}
        name_val = path
        if NAME_BY_VIP and base:
            vip = base.replace("https://","").replace("http://","")
            name_val = f"{vip}{path}"
        self.client.get(url,
                        headers=headers,
                        verify=VERIFY_TLS,
                        timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S),
                        name=name_val)
PY

chown -R opc:opc /home/opc/locustwork
echo READY
"""

# --------- terraform.tfvars from environment ---------
def _to_int(v: str, default: int) -> int:
    try:
        return int(float(v))
    except Exception:
        return default

tfvars = textwrap.dedent(f"""
oci_profile       = "{OCI_PROFILE}"
region            = "{REGION}"
compartment_id    = "{COMPARTMENT_ID}"
ad_a              = "{AD_A}"

enable_oke        = {"true" if ENABLE_OKE else "false"}
enable_generators = {"true" if ENABLE_GENERATORS else "false"}

cp_cidr              = "{CP_CIDR}"
pub_lb_cidr          = "{PUB_LB_CIDR}"
workers_private_cidr = "{WORKERS_PRIVATE_CIDR}"
pods_cidr            = "{PODS_CIDR}"
gens_pub_cidr        = "{GENS_PUB_CIDR}"

kubernetes_version      = "{KUBERNETES_VERSION}"
ssh_public_key_path_oke = "{SSH_PUBLIC_KEY_PATH_OKE}"
backend_node_shape      = "{BACKEND_NODE_SHAPE}"
backend_ocpus           = {_to_int(BACKEND_OCPUS, 16)}
backend_memory_gb       = {_to_int(BACKEND_MEMORY_GB, 64)}
max_pods_per_node       = {_to_int(MAX_PODS_PER_NODE, 31)}
backend_pool_size       = {_to_int(BACKEND_POOL_SIZE, 1)}
node_image_ocid_backend = "{NODE_IMAGE_OCID_BACKEND}"

# NEW — pass node pool initial labels
node_label_key          = "{NODE_LABEL_KEY}"
node_label_value        = "{NODE_LABEL_VALUE}"

generator_count        = {_to_int(GENERATOR_COUNT, 1)}
generator_shape        = "{GENERATOR_SHAPE}"
generator_ocpus        = {_to_int(GENERATOR_OCPUS, 8)}
generator_memory_gb    = {_to_int(GENERATOR_MEMORY_GB, 32)}
enable_ipv6_on_generators = {"true" if ENABLE_IPV6_ON_GENERATORS else "false"}
generator_image_id     = "{GEN_IMAGE_ID}"
ssh_public_key_content_gen = "{(GEN_SSH_PUBLIC_KEY_CONTENT or '').replace('"','\\"')}"
ssh_private_key_path_gen   = "{GEN_SSH_PRIVATE_KEY_PATH}"
""").strip("\n")

# --------- Write files ---------
files = {
    stack_dir / "provider.tf": provider_tf,
    stack_dir / "variables.tf": variables_tf,
    stack_dir / "locals.tf": locals_tf,
    stack_dir / "network.tf": network_tf,
    stack_dir / "oke.tf": oke_tf,
    stack_dir / "generators.tf": generators_tf,
    stack_dir / "outputs.tf": outputs_tf,
    stack_dir / "terraform.tfvars": tfvars,
    cloud_init_dir / "generator.sh": generator_cloud_init,
}

for path, content in files.items():
    path.write_text(content, encoding="utf-8")

print(f"Cell 4 complete: Wrote Terraform stack and cloud-init under: {stack_dir.resolve()}")
print("Files:")
for p in files.keys():
    print(" -", p)

print("\nNotes:")
print(" - COMPLETE STATELESS posture: Default Security List and all NSGs include symmetric protocol=all stateless rules for IPv4 and IPv6 (egress+ingress).")
print(" - Private subnets (workers_private, pods) use NAT for IPv4 only (no ::/0); public subnets route v4+v6 via IGW.")
print(" - OKE and Generators are conditionally created via enable_oke / enable_generators (from DEPLOY_MODE).")
print(" - Generator instances use generator_image_id from Cell 3 (shape-compatible Oracle Linux image).")
print(" - Direct Pod Routing is implemented later in Cell 7 (K8s manifests).")
print(" - NEW: NSG OCIDs are exported (nsg_lb_id, nsg_backends_id, nsg_generators_id) for Cell 7 to consume deterministically.")
print(" - NEW: Backend node pool will set initial_node_labels on creation (top-level): "
      f"{NODE_LABEL_KEY}={NODE_LABEL_VALUE} (via variables.tf/terraform.tfvars).")
print(" - Next: Run Cell 5 to terraform init/plan/apply and capture outputs (including NSG IDs).")


In [ ]:
# Cell 5 — Objective: Terraform init/validate/plan/apply for ./stack + capture outputs
# - Honors RUN_INFRA_APPLY (skips if false)
# - Optional pre-destroy if any DESTROY_*_ON_APPLY flags are true (full destroy, then fresh apply)
# - Cleans .terraform/ and lock file to avoid stale state
# - Saves terraform outputs to stack-outputs.json and a timestamped copy under OUTPUT_DIR
# - Persists GEN_IPS_V4_JSON and GEN_IPS_V6_JSON to the environment for later cells

import os, json, subprocess, shlex, time
from pathlib import Path
from datetime import datetime, timezone

def _fail(msg: str):
    raise RuntimeError(msg)

def _run(cmd: str, cwd: Path | None = None, check: bool = True) -> int:
    print(f"\n$ {cmd}")
    rc = subprocess.run(cmd, shell=True, cwd=str(cwd) if cwd else None).returncode
    if check and rc != 0:
        _fail(f"Command failed ({rc}): {cmd}")
    return rc

def _run_capture(cmd: str, cwd: Path | None = None, check: bool = True) -> tuple[int, str, str]:
    print(f"\n$ {cmd}")
    p = subprocess.run(cmd, shell=True, cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if check and p.returncode != 0:
        print((p.stdout or "").strip())
        print((p.stderr or "").strip())
        _fail(f"Command failed ({p.returncode}): {cmd}")
    return p.returncode, (p.stdout or ""), (p.stderr or "")

# Toggles and context from Cell 2/3
RUN_INFRA_APPLY       = os.environ.get("RUN_INFRA_APPLY", "true").strip().lower() in ("1","true","yes","y")
DESTROY_GENS_ON_APPLY = os.environ.get("DESTROY_GENS_ON_APPLY", "false").strip().lower() in ("1","true","yes","y")
DESTROY_OKE_ON_APPLY  = os.environ.get("DESTROY_OKE_ON_APPLY",  "false").strip().lower() in ("1","true","yes","y")
DESTROY_NETWORK_ON_APPLY = os.environ.get("DESTROY_NETWORK_ON_APPLY", "false").strip().lower() in ("1","true","yes","y")

OUTPUT_DIR = Path(os.path.abspath(os.environ.get("OUTPUT_DIR", "./results")))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TS_UTC = os.environ.get("TS_UTC", "") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
os.environ["TS_UTC"] = TS_UTC

# Workdir
stack_dir = Path("stack").resolve()
if not stack_dir.is_dir():
    _fail(f"Terraform working directory not found: {stack_dir}")

# Required files sanity
required = [
    "provider.tf", "variables.tf", "locals.tf",
    "network.tf", "oke.tf", "generators.tf", "outputs.tf",
    "terraform.tfvars"
]
missing = [f for f in required if not (stack_dir / f).exists()]
if missing:
    _fail(f"Missing Terraform files in {stack_dir}: {missing}")

# Optional: quick check for terraform binary (informative)
try:
    import shutil
    if shutil.which("terraform") is None:
        print("Warning: 'terraform' not found in PATH. Install Terraform and re-run this cell.")
except Exception:
    pass

if not RUN_INFRA_APPLY:
    print("RUN_INFRA_APPLY is false — skipping Terraform init/plan/apply.")
else:
    # If any DESTROY_*_ON_APPLY flags are set, perform a full destroy before apply
    if DESTROY_GENS_ON_APPLY or DESTROY_OKE_ON_APPLY or DESTROY_NETWORK_ON_APPLY:
        print("One or more DESTROY_*_ON_APPLY flags are true — performing a full terraform destroy before re-apply.")
        _run("terraform init -upgrade -input=false", cwd=stack_dir, check=True)
        _run("terraform destroy -parallelism=20 -auto-approve -var-file=terraform.tfvars", cwd=stack_dir, check=True)

    # Clean artifacts to avoid stale state
    lockfile = stack_dir / ".terraform.lock.hcl"
    tf_dir   = stack_dir / ".terraform"
    if lockfile.exists():
        print(f"Removing {lockfile}")
        try: lockfile.unlink()
        except Exception as e:
            print(f"Note: could not remove lockfile: {e}")
    if tf_dir.exists() and tf_dir.is_dir():
        print(f"Removing {tf_dir}")
        _run(f"rm -rf {shlex.quote(str(tf_dir))}", cwd=stack_dir, check=True)

    # Init with retry
    rc = _run("terraform init -upgrade -input=false", cwd=stack_dir, check=False)
    if rc != 0:
        print("Init failed; retrying in 5s ...")
        time.sleep(5)
        _run("terraform init -upgrade -input=false", cwd=stack_dir, check=True)

    # Validate
    _run("terraform validate", cwd=stack_dir, check=True)

    # Plan
    tfplan = "tfplan"
    _run(f"terraform plan -input=false -out={shlex.quote(tfplan)}", cwd=stack_dir, check=True)

    # Optional: preview first ~200 lines of plan
    print("\n--- Plan preview (first ~200 lines) ---")
    _run(f"terraform show -no-color {shlex.quote(tfplan)} | sed -n '1,200p'", cwd=stack_dir, check=False)

    # Apply
    _run(f"terraform apply -input=false -auto-approve {shlex.quote(tfplan)}", cwd=stack_dir, check=True)

    # Outputs to JSON + summary
    outputs_json = stack_dir / "stack-outputs.json"
    _run(f"terraform output -json > {shlex.quote(str(outputs_json))}", cwd=stack_dir, check=True)

    # Parse outputs
    outputs = {}
    try:
        outputs = json.loads(outputs_json.read_text(encoding="utf-8") or "{}")
    except Exception as e:
        print(f"Note: Could not parse outputs JSON. Reason: {e!r}")

    def _val(obj: dict, key: str):
        item = obj.get(key, {})
        if isinstance(item, dict) and "value" in item:
            return item.get("value")
        return item if item is not None else None

    VCN_ID = _val(outputs, "vcn_id")
    SUBNETS = _val(outputs, "subnets") or {}
    CLUSTER_ID = _val(outputs, "cluster_id")
    GEN_V4 = _val(outputs, "gen_public_ips") or []
    GEN_V6 = _val(outputs, "gen_ipv6_ips") or []

    # Persist to environment for downstream cells
    os.environ["GEN_IPS_V4_JSON"] = json.dumps(GEN_V4)
    os.environ["GEN_IPS_V6_JSON"] = json.dumps(GEN_V6)

    # Save a timestamped copy of outputs under OUTPUT_DIR
    ts_copy = OUTPUT_DIR / f"stack_outputs_{TS_UTC}.json"
    try:
        ts_copy.write_text(json.dumps({
            "vcn_id": VCN_ID,
            "subnets": SUBNETS,
            "cluster_id": CLUSTER_ID,
            "gen_public_ips": GEN_V4,
            "gen_ipv6_ips": GEN_V6,
            "ts_utc": TS_UTC
        }, indent=2), encoding="utf-8")
    except Exception as e:
        print(f"Note: Could not write timestamped outputs file. Reason: {e!r}")

    # Summary
    print("\n--- Terraform outputs (summary) ---")
    print("vcn_id:", VCN_ID)
    print("subnets:", SUBNETS)
    print("cluster_id:", CLUSTER_ID)
    print("gen_public_ips (IPv4):", GEN_V4)
    print("gen_ipv6_ips (IPv6):", GEN_V6)
    print(f"\nFull outputs saved to: {outputs_json}")
    if ts_copy.exists():
        print(f"Timestamped copy saved to: {ts_copy}")

print("\nCell 5 complete.")
print("NEXT:")
print(" - If ENABLE_OKE and RUN_K8S_DEPLOY=true: proceed to Cell 6 (kubeconfig/namespace/TLS secret).")
print(" - If ENABLE_GENERATORS and RUN_GENERATORS_PREP=true: after Cell 6/7, you will run Cell 9 to prepare generators.")
print(" - With RUN_LOCUST=true and no explicit targets, Cell 10 will auto-target the discovered OKE VIP when available.")


In [ ]:
# Cell 6 — Objective: Configure kubeconfig (if possible), create namespace, and provision TLS secret for OKE Service
# - Reads cluster_id from ./stack/stack-outputs.json
# - If 'oci' CLI is available: fetch kubeconfig for the OKE cluster (public endpoint)
# - Verifies kubectl context with version calls that avoid unsupported flags
# - Creates/ensures namespace (idempotent)
# - Generates a self‑signed TLS cert/key (RSA-2048 by default) and (re)creates secret 'ssl-certificate-secret'
# - Honors RUN_K8S_DEPLOY and ENABLE_OKE; skips with reason if disabled
# - v1.02 note: TLS secret is used in Cell 7 with direct pod routing (pods-as-backends)

import os, subprocess, shlex, json
from pathlib import Path

def _run(cmd: str, check: bool = True, cwd: Path | None = None) -> tuple[int, str, str]:
    print(f"\n$ {cmd}")
    p = subprocess.run(cmd, shell=True, text=True, cwd=str(cwd) if cwd else None,
                       capture_output=True)
    out, err = (p.stdout or "").strip(), (p.stderr or "").strip()
    if out:
        print(out)
    # Print stderr on error/fail text to aid troubleshooting
    if err and (("error" in err.lower()) or ("failed" in err.lower())):
        print(err)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {cmd}\n{out}\n{err}")
    return p.returncode, out, err

def _which(name: str) -> bool:
    try:
        import shutil
        return shutil.which(name) is not None
    except Exception:
        return False

def _load_stack_outputs() -> dict:
    p = Path("stack/stack-outputs.json").resolve()
    if not p.exists():
        raise FileNotFoundError(f"stack-outputs.json not found at {p}")
    try:
        return json.loads(p.read_text(encoding="utf-8") or "{}")
    except json.JSONDecodeError as e:
        raise RuntimeError(f"Could not parse stack-outputs.json: {e}")

# Toggles and inputs from prior cells
RUN_K8S_DEPLOY  = os.environ.get("RUN_K8S_DEPLOY", "true").strip().lower() in ("1","true","yes","y")
DEPLOY_MODE     = os.environ.get("DEPLOY_MODE", "both").strip().lower()
ENABLE_OKE      = (DEPLOY_MODE in ("oke-only", "both"))
WORK_NS         = os.environ.get("WORK_NS", "cpstest").strip() or "cpstest"
REGION          = os.environ.get("REGION", "").strip()

# TLS selection (from Cell 2; forced RSA unless changed there)
TLS_KEY_ALGO    = os.environ.get("TLS_KEY_ALGO", "rsa").strip().lower()       # "rsa" | "ecdsa"
TLS_ECDSA_CURVE = os.environ.get("TLS_ECDSA_CURVE", "prime256v1").strip()     # used only if TLS_KEY_ALGO=ecdsa

# Informational: pods-as-backends is fixed in v1.02; no action here but printed for context
SERVICE_BACKEND_MODE = os.environ.get("SERVICE_BACKEND_MODE", "pods").strip().lower()

if not RUN_K8S_DEPLOY:
    print("RUN_K8S_DEPLOY is false — skipping kubeconfig/namespace/TLS secret setup.")
elif not ENABLE_OKE:
    print("ENABLE_OKE is false (DEPLOY_MODE != oke-only/both) — skipping kubeconfig/namespace/TLS secret setup.")
else:
    # Load cluster_id from Terraform outputs
    outs = _load_stack_outputs()
    cluster_id = (outs.get("cluster_id") or {}).get("value") if isinstance(outs.get("cluster_id"), dict) else outs.get("cluster_id")
    if not cluster_id:
        raise RuntimeError("cluster_id not found in stack-outputs.json. Ensure Cell 5 completed successfully.")
    print(f"Cluster ID: {cluster_id}")

    # 1) Kubeconfig setup if 'oci' CLI is present; otherwise print instructions
    if not REGION:
        raise RuntimeError("REGION is empty in environment. Ensure Cell 3 Apply persisted REGION.")
    if _which("oci"):
        home = Path.home()
        kube_dir = home / ".kube"
        kube_dir.mkdir(parents=True, exist_ok=True)
        kubeconfig_path = kube_dir / "config"
        print(f"Attempting to write kubeconfig to: {kubeconfig_path}")
        _run(
            f"oci ce cluster create-kubeconfig "
            f"--cluster-id {shlex.quote(cluster_id)} "
            f"--file {shlex.quote(str(kubeconfig_path))} "
            f"--region {shlex.quote(REGION)} "
            f"--token-version 2.0.0 "
            f"--kube-endpoint PUBLIC_ENDPOINT "
            f"--overwrite",
            check=True
        )
        _run("kubectl version --client", check=False)
        _run("kubectl version", check=False)
        _run("kubectl config current-context", check=False)
    else:
        print("oci CLI not found — Skipping automatic kubeconfig setup.")
        print("Manual option (from your shell):")
        print(f"  oci ce cluster create-kubeconfig --cluster-id {cluster_id} "
              f"--file ~/.kube/config --region {REGION} --token-version 2.0.0 "
              f"--kube-endpoint PUBLIC_ENDPOINT --overwrite")
        print("After that, verify with:")
        print("  kubectl version --client")
        print("  kubectl version")
        print("  kubectl config current-context")

    # 2) Create namespace (idempotent)
    print(f"\nEnsuring namespace exists: {WORK_NS}")
    rc, out, err = _run(f"kubectl get ns {shlex.quote(WORK_NS)}", check=False)
    if rc != 0:
        _run(f"kubectl create namespace {shlex.quote(WORK_NS)}", check=True)
    else:
        print(f"Namespace {WORK_NS} already exists.")

    # 3) Generate TLS cert/key (RSA-2048 by default) and (re)create secret
    tls_key = Path("tls.key").resolve()
    tls_crt = Path("tls.crt").resolve()

    if TLS_KEY_ALGO == "ecdsa":
        print(f"\nGenerating self-signed TLS certificate (ECDSA curve={TLS_ECDSA_CURVE}, 365 days) ...")
        _run(f"openssl ecparam -name {shlex.quote(TLS_ECDSA_CURVE)} -genkey -noout -out {shlex.quote(str(tls_key))}", check=True)
        _run(
            "openssl req -x509 -new "
            f"-key {shlex.quote(str(tls_key))} "
            "-sha256 -days 365 "
            "-subj \"/CN=nginxsvc/O=nginxsvc\" "
            f"-out {shlex.quote(str(tls_crt))}",
            check=True
        )
    else:
        print("\nGenerating self-signed TLS certificate (RSA:2048, 365 days) ...")
        _run(
            "openssl req -x509 -nodes -days 365 -newkey rsa:2048 "
            f"-keyout {shlex.quote(str(tls_key))} "
            f"-out {shlex.quote(str(tls_crt))} "
            "-subj \"/CN=nginxsvc/O=nginxsvc\"",
            check=True
        )

    # Optional: verify cert key type (informational)
    _run(f"openssl x509 -in {shlex.quote(str(tls_crt))} -noout -text | grep -E 'Public Key Algorithm|Curve|RSA Public-Key' || true", check=False)

    secret_name = "ssl-certificate-secret"
    print(f"\n(Re)creating TLS secret '{secret_name}' in namespace '{WORK_NS}' ...")
    _run(f"kubectl -n {shlex.quote(WORK_NS)} delete secret {shlex.quote(secret_name)} --ignore-not-found", check=False)
    _run(
        f"kubectl -n {shlex.quote(WORK_NS)} create secret tls {shlex.quote(secret_name)} "
        f"--key {shlex.quote(str(tls_key))} --cert {shlex.quote(str(tls_crt))}",
        check=True
    )
    _run(f"kubectl -n {shlex.quote(WORK_NS)} get secret {shlex.quote(secret_name)} -o yaml | head -n 20", check=False)

    # 4) Summary and next steps
    print("\nCell 6 complete:")
    print(f" - kubeconfig: {'configured via oci CLI' if _which('oci') else 'not configured (instructions printed)'}")
    print(f" - namespace: {WORK_NS} present")
    print(f" - secret: {secret_name} created in ns/{WORK_NS} (key_algo={TLS_KEY_ALGO})")
    print(f" - info: pods-as-backends mode is '{SERVICE_BACKEND_MODE}' (enforced in Cell 7)")
    print("\nNEXT:")
    print(" - Run Cell 7 to deploy the backend tuner, NGINX DaemonSet (no hostNetwork), and the LoadBalancer Service with")
    print("   pods-as-backends annotations, then validate the VIP.")


In [ ]:
# Cell 7 — Objective: Seamless Pods‑as‑Backends deploy (Security Lists mode) with automatic fallback (no manual intervention)
# - Writes and applies:
#   1) ds-backend-tuner.yaml  (hostNetwork removed; privileged+hostPID retained)
#   2) cm_nginx.yaml          (30080 with proxy_protocol, 30081 plain; /healthz=200)
#   3) cps_deployment.yaml    (DaemonSet + Service)
#        Service CREATE‑TIME annotations:
#          * pods‑as‑backends (PRIMARY + BETA)
#          * PPv2 (PRIMARY + BETA)
#          * Security Lists: "None" (PRIMARY + BETA)
#          * Health‑check: TCP:30080 (aligned to backend/data port)
#        (No NSG annotations.)
#   4) nginx-extra-deployment.yaml (Deployment) → replicas = (#backend nodes × NGINX_PODS_PER_NODE)
# - Flow:
#   * Create DS/CM/Service → wait VIP → nudge CCM → poll membership
#   * If membership = 0 AND Endpoints > 0: automatically DELETE ONLY the Service and recreate it (clean Create path)
#   * Validate VIP /healthz (expect 200)
#
# Run AFTER Cell 6 (kubeconfig, namespace, TLS secret)
# Inputs:
#   WORK_NS                (default "cpstest")
#   NGINX_PODS_PER_NODE    (default "7" for 16 OCPU/64GB)
#   COMPARTMENT_ID, REGION (optional, for OCI LB membership polling)

import os, json, shlex, subprocess, time, re, textwrap
from pathlib import Path

NS  = os.environ.get("WORK_NS", "cpstest").strip() or "cpstest"
SVC = "nginx-service"
PODS_PER_NODE = int(os.environ.get("NGINX_PODS_PER_NODE", "0"))

def q(s: str) -> str:
    return shlex.quote(s)

def run(cmd: str, check=False, quiet=False):
    if not quiet: print(f"\n$ {cmd}")
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    out, err = (p.stdout or "").strip(), (p.stderr or "").strip()
    if not quiet and out: print(out)
    if not quiet and err and any(w in err.lower() for w in ["error","failed","exception"]): print(err)
    if check and p.returncode != 0: raise RuntimeError(f"Command failed ({p.returncode}): {cmd}\n{out}\n{err}")
    return p.returncode, out, err

def get_json(cmd: str):
    rc, out, _ = run(cmd, quiet=True)
    try: return json.loads(out) if (rc==0 and out) else {}
    except Exception: return {}

def endpoints_count(ns: str, svc: str) -> int:
    e = get_json(f"kubectl -n {q(ns)} get endpoints {q(svc)} -o json")
    try:
        subs = e.get("subsets", []) or []
        return sum(len(s.get("addresses", []) or []) for s in subs)
    except Exception:
        return -1

def discover_vip(ns: str, svc: str, wait_seconds: int = 300) -> str:
    vip = ""
    deadline = time.time() + wait_seconds
    while time.time() < deadline and not vip:
        rc, vip_ip, _  = run(f"kubectl -n {q(ns)} get svc {q(svc)} -o jsonpath='{{.status.loadBalancer.ingress[0].ip}}'", quiet=True)
        rc2, vip_dns, _= run(f"kubectl -n {q(ns)} get svc {q(svc)} -o jsonpath='{{.status.loadBalancer.ingress[0].hostname}}'", quiet=True)
        vip = (vip_ip or vip_dns).strip()
        if not vip:
            print("Waiting for EXTERNAL-IP ...")
            time.sleep(5)
    print("\nVIP:", vip or "(pending)")
    return vip

def poll_membership_by_vip(vip: str, comp: str, region: str, timeout_s: int = 180) -> tuple[bool, str]:
    # Returns (any_members, lb_id) best-effort using OCI CLI
    rc, _, _ = run("which oci", quiet=True)
    if rc != 0 or not vip or not comp or not region:
        return (False, "")
    lb_id = ""
    rc, lbs_out, _ = run(f"oci lb load-balancer list --compartment-id {q(comp)} --all --region {q(region)} --output json 2>/dev/null", quiet=True)
    try:
        for item in (json.loads(lbs_out).get("data") or []):
            for addr in (item.get("ip-addresses") or []):
                if (addr.get("ip-address") or "") == vip: lb_id = item.get("id") or ""; break
            if lb_id: break
    except Exception: pass
    print("LB OCID:", lb_id or "(not found)")
    if not lb_id:
        return (False, "")
    end = time.time() + timeout_s
    any_members = False
    while time.time() < end and not any_members:
        rc, bs_out, _ = run(f"oci lb backend-set list --load-balancer-id {q(lb_id)} --all --region {q(region)} --output json 2>/dev/null", quiet=True)
        try:
            for b in (json.loads(bs_out).get("data") or []):
                name=b.get("name")
                rc, be_out, _ = run(f"oci lb backend list --load-balancer-id {q(lb_id)} --backend-set-name {q(name)} --all --region {q(region)} --output json 2>/dev/null", quiet=True)
                backs = (json.loads(be_out).get("data") or [])
                if backs:
                    print("Members:", json.dumps([{ "ip":m.get("ip-address"), "port":m.get("port"), "status":m.get("status")} for m in backs]))
                    any_members = True
                    break
        except Exception: pass
        if not any_members:
            print("No members yet; waiting ...")
            time.sleep(5)
    return (any_members, lb_id)

print(f"Namespace: {NS}  NGINX_PODS_PER_NODE: {PODS_PER_NODE}")

# 0) Sanity: TLS secret must exist (Cell 6)
rc, _, _ = run(f"kubectl -n {q(NS)} get secret ssl-certificate-secret -o name", quiet=True)
if rc != 0:
    raise SystemExit(f"TLS secret 'ssl-certificate-secret' not found in ns/{NS}. Re-run Cell 6 before this cell.")

# 1) ds-backend-tuner.yaml (hostNetwork removed; privileged+hostPID retained)
tuner_yaml = textwrap.dedent(f"""
apiVersion: apps/v1
kind: DaemonSet
metadata:
  name: ds-backend-tuner
  namespace: {NS}
spec:
  selector:
    matchLabels:
      app: backend-tuner
      stage: final
  template:
    metadata:
      labels:
        app: backend-tuner
        stage: final
    spec:
      hostPID: true
      serviceAccountName: default
      nodeSelector:
        app.role: backend
      tolerations:
      - operator: "Exists"
      containers:
      - name: tuner
        image: oraclelinux:8-slim
        imagePullPolicy: IfNotPresent
        securityContext:
          privileged: true
          allowPrivilegeEscalation: true
          readOnlyRootFilesystem: false
        resources:
          requests: {{ cpu: "10m", memory: "32Mi" }}
          limits:   {{ cpu: "100m", memory: "128Mi" }}
        command: ["/bin/sh","-c"]
        args:
        - |
          set -euo pipefail
          SYSCTL_BIN="/sbin/sysctl"
          if ! nsenter -t 1 -m -u -i -n -p -- test -x "$SYSCTL_BIN"; then
            if nsenter -t 1 -m -u -i -n -p -- test -x /usr/sbin/sysctl; then
              SYSCTL_BIN="/usr/sbin/sysctl"
            else
              SYSCTL_BIN="sysctl"
            fi
          fi
          nsenter -t 1 -m -u -i -n -p -- "$SYSCTL_BIN" -w net.core.somaxconn=262144
          nsenter -t 1 -m -u -i -n -p -- "$SYSCTL_BIN" -w net.core.netdev_max_backlog=500000
          nsenter -t 1 -m -u -i -n -p -- "$SYSCTL_BIN" -w net.ipv4.tcp_max_syn_backlog=262144
          nsenter -t 1 -m -u -i -n -p -- "$SYSCTL_BIN" -w net.ipv4.ip_local_port_range="1024 65535"
          nsenter -t 1 -m -u -i -n -p -- "$SYSCTL_BIN" -w net.ipv4.tcp_fin_timeout=15
          echo "Host values after apply:"
          nsenter -t 1 -m -u -i -n -p -- "$SYSCTL_BIN" -n net.core.somaxconn || true
          nsenter -t 1 -m -u -i -n -p -- "$SYSCTL_BIN" -n net.core.netdev_max_backlog || true
          nsenter -t 1 -m -u -i -n -p -- "$SYSCTL_BIN" -n net.ipv4.tcp_max_syn_backlog || true
          nsenter -t 1 -m -u -i -n -p -- "$SYSCTL_BIN" -n net.ipv4.ip_local_port_range || true
          nsenter -t 1 -m -u -i -n -p -- "$SYSCTL_BIN" -n net.ipv4.tcp_fin_timeout || true
          echo "Final Stage D sysctls applied."
          sleep infinity
        volumeMounts:
        - name: nsenter
          mountPath: /usr/bin/nsenter
          readOnly: true
      volumes:
      - name: nsenter
        hostPath: {{ path: /usr/bin/nsenter, type: File }}
""").strip("\n")
Path("ds-backend-tuner.yaml").write_text(tuner_yaml, encoding="utf-8")
run("kubectl apply -f ds-backend-tuner.yaml", check=True)

print("\nWaiting for ds-backend-tuner to be Ready (up to 120s) ...")
deadline = time.time() + 120
ready=False
while time.time()<deadline and not ready:
    ds = get_json(f"kubectl -n {q(NS)} get ds ds-backend-tuner -o json")
    try:
        st = ds.get("status") or {}
        desired = int(st.get("desiredNumberScheduled") or 0)
        rdy = int(st.get("numberReady") or 0)
        print(f"Desired={desired} Ready={rdy}")
        ready = desired>0 and rdy>=desired
    except Exception: pass
    if not ready: time.sleep(4)
run(f"kubectl -n {q(NS)} get ds ds-backend-tuner -o wide")

print("\nAllowing node settle time (sleep 45s) ...")
time.sleep(45)

# 2) cm_nginx.yaml (30080 with proxy_protocol; 30081 plain; /healthz=200)
nginx_conf = """user  nginx;
worker_processes  auto;
worker_rlimit_nofile 1048576;

events {
    worker_connections  131072;
    multi_accept on;
    accept_mutex off;
}

http {
    include       /etc/nginx/mime.types;
    default_type  application/octet-stream;

    sendfile on;
    tcp_nopush on;
    tcp_nodelay on;

    keepalive_timeout 15;
    client_body_timeout 10;
    client_header_timeout 10;
    send_timeout 10;
    reset_timedout_connection on;

    access_log off;
    server_tokens off;

    server {
        listen       30080 proxy_protocol reuseport backlog=262144;
        listen       30081 reuseport backlog=262144;
        server_name  localhost;

        location /healthz {
            return 200 "ok\\n";
        }

        location /payload_ {
            root /usr/share/nginx/html;
        }

        location / {
            return 200 "Client IP: $proxy_protocol_addr\\n";
        }
    }
}"""
indented_conf = "\n".join("    " + ln for ln in nginx_conf.splitlines())
cm_yaml = f"""apiVersion: v1
kind: ConfigMap
metadata:
  name: nginx-config
  namespace: {NS}
data:
  nginx.conf: |
{indented_conf}
"""
Path("cm_nginx.yaml").write_text(cm_yaml, encoding="utf-8")
run("kubectl apply -f cm_nginx.yaml", check=True)

# 3) cps_deployment.yaml (DaemonSet + Service) with readinessProbe and SL mode (pods-as-backends/PPv2 + TCP:30080 checker at CREATE time)
svc_annotations_block = """    oci.oraclecloud.com/load-balancer-type: "lb"
    service.beta.kubernetes.io/oci-load-balancer-shape: "flexible"
    service.beta.kubernetes.io/oci-load-balancer-shape-flex-min: "8000"
    service.beta.kubernetes.io/oci-load-balancer-shape-flex-max: "8000"
    service.beta.kubernetes.io/oci-load-balancer-ssl-ports: "443"
    service.beta.kubernetes.io/oci-load-balancer-tls-secret: "ssl-certificate-secret"
    service.beta.kubernetes.io/oci-load-balancer-backend-protocol: "TCP"
    # PPv2 (PRIMARY + BETA)
    oci.oraclecloud.com/load-balancer-connection-proxy-protocol-version: "2"
    service.beta.kubernetes.io/oci-load-balancer-connection-proxy-protocol-version: "2"
    # Pods-as-backends (PRIMARY + BETA)
    oci.oraclecloud.com/load-balancer-backend-type: "ip"
    service.beta.kubernetes.io/oci-load-balancer-backend-type: "ip"
    # Security Lists (PRIMARY + BETA)
    oci.oraclecloud.com/security-rule-management-mode: "None"
    service.beta.kubernetes.io/oci-load-balancer-security-list-management-mode: "None"
    # Health-check aligned to backend port (TCP:30080)
    oci-load-balancer.oraclecloud.com/health-check: |
      {
        "protocol": "TCP",
        "port": 30080,
        "intervalInMillis": 10000,
        "timeoutInMillis": 3000,
        "retries": 3
      }"""

ds_svc_yaml = f"""apiVersion: apps/v1
kind: DaemonSet
metadata:
  name: nginx-backend
  namespace: {NS}
  labels:
    app: nginx
spec:
  selector:
    matchLabels:
      app: nginx
  template:
    metadata:
      labels:
        app: nginx
    spec:
      nodeSelector:
        app.role: backend
      initContainers:
      - name: bake-payloads
        image: busybox:1.36
        imagePullPolicy: IfNotPresent
        command: ["/bin/sh","-c"]
        args:
        - |
          set -e
          dd if=/dev/zero of=/payloads/payload_4k   bs=1000 count=4   status=none
          dd if=/dev/zero of=/payloads/payload_10k  bs=1000 count=10  status=none
          dd if=/dev/zero of=/payloads/payload_50k  bs=1000 count=50  status=none
          dd if=/dev/zero of=/payloads/payload_100k bs=1000 count=100 status=none
          dd if=/dev/zero of=/payloads/payload_256k bs=1000 count=256 status=none
          dd if=/dev/zero of=/payloads/payload_1m   bs=1000000 count=1 status=none
          ls -l /payloads
        volumeMounts:
        - name: payloads
          mountPath: /payloads
      containers:
      - name: nginx
        image: index.docker.io/library/nginx:latest
        imagePullPolicy: IfNotPresent
        ports:
        - containerPort: 30080
        - containerPort: 30081
        resources:
          requests:
            cpu: "1000m"
            memory: "512Mi"
        readinessProbe:
          httpGet:
            path: /healthz
            port: 30081
          initialDelaySeconds: 2
          periodSeconds: 3
          timeoutSeconds: 1
          failureThreshold: 3
        volumeMounts:
        - name: config
          mountPath: /etc/nginx/nginx.conf
          subPath: nginx.conf
        - name: payloads
          mountPath: /usr/share/nginx/html
      volumes:
      - name: config
        configMap:
          name: nginx-config
      - name: payloads
        emptyDir: {{}}
---
apiVersion: v1
kind: Service
metadata:
  name: {SVC}
  namespace: {NS}
  annotations:
{svc_annotations_block}
spec:
  type: LoadBalancer
  allocateLoadBalancerNodePorts: false
  selector:
    app: nginx
  ports:
  - name: https
    port: 443
    targetPort: 30080
    protocol: TCP
"""
Path("cps_deployment.yaml").write_text(ds_svc_yaml, encoding="utf-8")
run("kubectl apply -f cps_deployment.yaml", check=True)

# 4) nginx-extra-deployment.yaml (Deployment) — replicas = (#backend nodes × PODS_PER_NODE)
rc, nodes_out, _ = run("kubectl get nodes -l app.role=backend -o name", check=True)
backend_nodes = len([ln for ln in (nodes_out.splitlines() if nodes_out else []) if ln.strip()])
replicas = max(0, backend_nodes * PODS_PER_NODE)
print(f"Backend nodes detected: {backend_nodes}  -> Extra deployment replicas: {replicas}")

extra_dep_yaml = f"""apiVersion: apps/v1
kind: Deployment
metadata:
  name: nginx-backend-extra
  namespace: {NS}
  labels:
    app: nginx
    role: extra
spec:
  replicas: {replicas}
  selector:
    matchLabels:
      app: nginx
      role: extra
  template:
    metadata:
      labels:
        app: nginx
        role: extra
    spec:
      nodeSelector:
        app.role: backend
      topologySpreadConstraints:
      - maxSkew: 1
        topologyKey: kubernetes.io/hostname
        whenUnsatisfiable: ScheduleAnyway
        labelSelector:
          matchLabels:
            app: nginx
      containers:
      - name: nginx
        image: index.docker.io/library/nginx:latest
        imagePullPolicy: IfNotPresent
        ports:
        - containerPort: 30080
        - containerPort: 30081
        resources:
          requests:
            cpu: "1000m"
            memory: "512Mi"
        readinessProbe:
          httpGet:
            path: /healthz
            port: 30081
          initialDelaySeconds: 2
          periodSeconds: 3
          timeoutSeconds: 1
          failureThreshold: 3
        volumeMounts:
        - name: config
          mountPath: /etc/nginx/nginx.conf
          subPath: nginx.conf
        - name: payloads
          mountPath: /usr/share/nginx/html
      volumes:
      - name: config
        configMap:
          name: nginx-config
      - name: payloads
        emptyDir: {{}}
"""
Path("nginx-extra-deployment.yaml").write_text(extra_dep_yaml, encoding="utf-8")
run("kubectl apply -f nginx-extra-deployment.yaml", check=True)

# 5) Rollouts and local health check
run(f"kubectl -n {q(NS)} rollout restart ds/nginx-backend", check=True)
run(f"kubectl -n {q(NS)} rollout status ds/nginx-backend --timeout=300s", check=True)
run(f"kubectl -n {q(NS)} rollout status deploy/nginx-backend-extra --timeout=600s", check=True)

rc, pods, _ = run(f"kubectl -n {q(NS)} get pods -l app=nginx -o name", check=True)
sample = (pods.splitlines() or [""])[0]
if sample:
    run(f"kubectl -n {q(NS)} exec {sample} -- sh -lc 'curl -sSI --max-time 3 http://127.0.0.1:30081/healthz || true'")

# 6) Defensive cleanup: remove legacy/NSG annotations (idempotent; DO NOT delete Service)
run(f"kubectl -n {q(NS)} annotate svc {SVC} oci.oraclecloud.com/node-label-selector- --overwrite", check=False)
run(f"kubectl -n {q(NS)} annotate svc {SVC} oci.oraclecloud.com/oci-backend-network-security-group- --overwrite", check=False)
run(f"kubectl -n {q(NS)} annotate svc {SVC} oci.oraclecloud.com/load-balancer-security-groups- --overwrite", check=False)
run(f"kubectl -n {q(NS)} annotate svc {SVC} oci-load-balancer.oraclecloud.com/security-groups- --overwrite", check=False)

# 7) VIP + nudge CCM
vip = discover_vip(NS, SVC, wait_seconds=300)
if not vip:
    raise SystemExit("EXTERNAL-IP still pending; re-run later.")
run(f"kubectl -n {q(NS)} annotate svc {SVC} oci.oraclecloud.com/oci-ccm-reconcile-ts='{int(time.time())}' --overwrite", check=True)

# 8) Poll membership
COMP = os.environ.get("COMPARTMENT_ID","").strip()
REG  = os.environ.get("REGION","").strip()
def derive_region_from_cluster_ocid() -> str:
    p = Path("stack/stack-outputs.json")
    if not p.is_file(): return ""
    try:
        obj = json.loads(p.read_text(encoding="utf-8") or "{}")
        cid = obj.get("cluster_id", {}); cid = cid.get("value") if isinstance(cid, dict) else cid
        m = re.search(r"^ocid1\\.cluster\\.oc1\\.([^.]+)\\.", (cid or ""))
        return m.group(1) if m else ""
    except Exception: return ""
if not REG:
    r = derive_region_from_cluster_ocid()
    if r: REG = r; print("REGION (derived):", REG)

eps_cnt = endpoints_count(NS, SVC)
print("Endpoints addresses:", eps_cnt)

any_members, lb_id = poll_membership_by_vip(vip, COMP, REG, timeout_s=180)

# 9) Automatic fallback: ONE-TIME Service recreate if endpoints>0 and still no members
if eps_cnt > 0 and not any_members:
    print("\nNo members after Update path. Performing one-time immutable-safe Service recreate ...")
    svc_only_yaml = f"""apiVersion: v1
kind: Service
metadata:
  name: {SVC}
  namespace: {NS}
  annotations:
{svc_annotations_block}
spec:
  type: LoadBalancer
  allocateLoadBalancerNodePorts: false
  selector:
    app: nginx
  ports:
  - name: https
    port: 443
    targetPort: 30080
    protocol: TCP
"""
    Path("svc_recreate.yaml").write_text(svc_only_yaml, encoding="utf-8")
    run(f"kubectl -n {q(NS)} delete svc {SVC} --ignore-not-found --wait=true", check=False)
    run("kubectl apply -f svc_recreate.yaml", check=True)
    vip = discover_vip(NS, SVC, wait_seconds=480)
    if not vip:
        raise SystemExit("EXTERNAL-IP still pending after recreate; re-run later.")
    run(f"kubectl -n {q(NS)} annotate svc {SVC} oci.oraclecloud.com/oci-ccm-reconcile-ts='{int(time.time())}' --overwrite", check=True)
    any_members, lb_id = poll_membership_by_vip(vip, COMP, REG, timeout_s=240)

# 10) Short settle; then validate VIP /healthz
print("\nAllowing LB convergence (sleep 45s) ...")
time.sleep(45)

print("\nVIP /healthz (expect 200):")
run(f"curl --http1.1 -skI -w '%{{http_code}}\\n' --connect-timeout 8 --max-time 12 https://{q(vip)}/healthz | head -n2")

print(f"\nDone. Service={SVC} ns={NS}, VIP={vip}, extra replicas={replicas} (nodes×{PODS_PER_NODE}).")


In [ ]:
# Cell 8 — Objective: Discover/persist OKE VIP, normalize targets SSOT for Locust/Generators, validate reachability
# - Discovers EXTERNAL-IP of Service nginx-service in WORK_NS and persists:
#   * Env: OKE_VIP, LOCUST_TARGETS, LOCUST_* (mode, verify_tls, timeouts, paths, name-by-vip)
#   * Files: results/oke_vip_<TS_UTC>.json, results/targets_<TS_UTC>.json, results/locust_env_<TS_UTC>.sh
# - Normalizes targets: combines OKE VIP base (https://VIP) with EXTERNAL_TARGETS_TEXT (if any), de-duplicates
# - Validates reachability:
#   * HEAD https://VIP/healthz (TLS verify per LOCUST_VERIFY_TLS)
#   * Optional GET https://VIP/ (prints first line, should include “Client IP: …” when PPv2 path is exercised)
# - Idempotent and safe: no cluster changes

import os, json, shlex, subprocess, time
from pathlib import Path
from datetime import datetime, timezone

def q(s: str) -> str:
    return shlex.quote(s)

def run(cmd: str, check=False, quiet=False):
    if not quiet: print(f"\n$ {cmd}")
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    out, err = (p.stdout or "").strip(), (p.stderr or "").strip()
    if not quiet and out: print(out)
    # print errors only when meaningful
    if not quiet and err and any(w in err.lower() for w in ["error","failed","exception"]):
        print(err)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {cmd}\n{out}\n{err}")
    return p.returncode, out, err

def discover_vip(ns: str, svc: str, wait_seconds: int = 300) -> str:
    vip = ""
    deadline = time.time() + wait_seconds
    while time.time() < deadline and not vip:
        rc, vip_ip, _  = run(f"kubectl -n {q(ns)} get svc {q(svc)} -o jsonpath='{{.status.loadBalancer.ingress[0].ip}}'", quiet=True)
        rc2, vip_dns, _= run(f"kubectl -n {q(ns)} get svc {q(svc)} -o jsonpath='{{.status.loadBalancer.ingress[0].hostname}}'", quiet=True)
        vip = (vip_ip or vip_dns).strip()
        if not vip:
            print("Waiting for EXTERNAL-IP ...")
            time.sleep(5)
    print("\nVIP:", vip or "(pending)")
    return vip

def _normalize_targets_csv(csv_text: str) -> list[str]:
    # Normalize list of hosts/URLs:
    # - Add https:// if scheme missing
    # - Leave ports/IPv4 as-is; bracket IPv6 literals if missing
    out = []
    for tok in (csv_text or "").split(","):
        t = (tok or "").strip()
        if not t:
            continue
        if not (t.startswith("http://") or t.startswith("https://")):
            t = "https://" + t
        try:
            scheme, rest = t.split("://", 1)
            host = rest.split("/")[0]
            # If IPv6 literal without brackets, add them
            if ":" in host and not (host.startswith("[") and host.endswith("]")):
                t = f"{scheme}://[{host}]" + rest[len(host):]
        except Exception:
            pass
        if t not in out:
            out.append(t)
    return out

def _ms_to_s(ms_env: str, default_s: float) -> float:
    try:
        return max(0.0, float(int(ms_env))/1000.0)
    except Exception:
        return float(default_s)

# Inputs/SSOT from previous cells
NS              = os.environ.get("WORK_NS", "cpstest").strip() or "cpstest"
SVC_NAME        = os.environ.get("SVC_NAME", "nginx-service").strip() or "nginx-service"
OUTPUT_DIR      = Path(os.path.abspath(os.environ.get("OUTPUT_DIR", "./results")))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TS_UTC          = os.environ.get("TS_UTC", "") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
os.environ["TS_UTC"] = TS_UTC

# Locust/Targets SSOT (from Cell 2/3)
TEST_MODE                 = os.environ.get("TEST_MODE", "cps").strip().lower()           # "cps" | "throughput"
LOCUST_VERIFY_TLS_BOOL    = os.environ.get("LOCUST_VERIFY_TLS", "false").strip().lower() in ("1","true","yes","y")
WAIT_TIME_SEC             = float(os.environ.get("LOCUST_WAIT_TIME_SEC", "1.0"))
CONNECT_TIMEOUT_S         = _ms_to_s(os.environ.get("LOCUST_CONNECT_TIMEOUT_MS", "8000"), 8.0)
READ_TIMEOUT_S            = _ms_to_s(os.environ.get("LOCUST_READ_TIMEOUT_MS",  "15000"), 15.0)
HEALTH_PATH               = os.environ.get("HEALTH_ENDPOINT_PATH", "/healthz").strip() or "/healthz"
THROUGHPUT_PATH           = os.environ.get("THROUGHPUT_ENDPOINT_PATH", "/payload_100k").strip() or "/payload_100k"
EXTERNAL_TARGETS_TEXT     = os.environ.get("EXTERNAL_TARGETS_TEXT", "").strip()

print(f"Namespace: {NS}  Service: {SVC_NAME}")
print(f"SSOT: mode={TEST_MODE} verify_tls={'true' if LOCUST_VERIFY_TLS_BOOL else 'false'} wait_s={WAIT_TIME_SEC} connect_s={CONNECT_TIMEOUT_S} read_s={READ_TIMEOUT_S}")
print(f"Paths: health={HEALTH_PATH} throughput={THROUGHPUT_PATH}")

# 1) Discover VIP
vip = discover_vip(NS, SVC_NAME, wait_seconds=300)
if not vip:
    raise SystemExit("EXTERNAL-IP still pending; re-run this cell later once the Service has a VIP.")

base_url = f"https://{vip}"
os.environ["OKE_VIP"] = vip

# 2) Normalize/compose targets: EXTERNAL_TARGETS_TEXT (optional) + OKE VIP base
external_targets = _normalize_targets_csv(EXTERNAL_TARGETS_TEXT)
# Always append the discovered OKE base host (not a path; Locust appends the path each request)
targets = [*external_targets, base_url]
# De-duplicate while preserving order
seen = set(); targets = [t for t in targets if (t not in seen and not seen.add(t))]

# Persist core Locust env expected by the generator locustfile
os.environ["LOCUST_MODE"]               = TEST_MODE
os.environ["LOCUST_VERIFY_TLS"]         = "true" if LOCUST_VERIFY_TLS_BOOL else "false"
os.environ["LOCUST_CONNECT_TIMEOUT_S"]  = f"{CONNECT_TIMEOUT_S:.3f}"
os.environ["LOCUST_READ_TIMEOUT_S"]     = f"{READ_TIMEOUT_S:.3f}"
os.environ["LOCUST_WAIT_TIME_S"]        = f"{WAIT_TIME_SEC:.3f}"
os.environ["LOCUST_HEALTH_PATH"]        = HEALTH_PATH
os.environ["LOCUST_THROUGHPUT_PATH"]    = THROUGHPUT_PATH
os.environ["LOCUST_TARGETS"]            = ",".join(targets)
os.environ["LOCUST_NAME_BY_VIP"]        = "true"  # convenient naming in reports

# 3) Write results artifacts
# 3a) VIP details file
vip_json_path = OUTPUT_DIR / f"oke_vip_{TS_UTC}.json"
svc_json_rc, svc_json_out, _ = run(f"kubectl -n {q(NS)} get svc {q(SVC_NAME)} -o json", check=False, quiet=True)
svc_obj = {}
try:
    svc_obj = json.loads(svc_json_out) if svc_json_out else {}
except Exception:
    svc_obj = {}

vip_rec = {
    "namespace": NS,
    "service_name": SVC_NAME,
    "vip": vip,
    "ts_utc": TS_UTC,
    "service": {
        "annotations": (svc_obj.get("metadata") or {}).get("annotations"),
        "ports": (svc_obj.get("spec") or {}).get("ports"),
        "type": (svc_obj.get("spec") or {}).get("type"),
        "selector": (svc_obj.get("spec") or {}).get("selector"),
    },
}
vip_json_path.write_text(json.dumps(vip_rec, indent=2), encoding="utf-8")
print(f"Wrote: {vip_json_path}")

# 3b) Targets file
targets_json_path = OUTPUT_DIR / f"targets_{TS_UTC}.json"
targets_rec = {
    "targets": targets,
    "mode": TEST_MODE,
    "verify_tls": LOCUST_VERIFY_TLS_BOOL,
    "connect_timeout_s": CONNECT_TIMEOUT_S,
    "read_timeout_s": READ_TIMEOUT_S,
    "wait_time_s": WAIT_TIME_SEC,
    "health_path": HEALTH_PATH,
    "throughput_path": THROUGHPUT_PATH,
    "ts_utc": TS_UTC
}
targets_json_path.write_text(json.dumps(targets_rec, indent=2), encoding="utf-8")
print(f"Wrote: {targets_json_path}")

# 3c) Portable env snippet for generators/locust
env_snip_path = OUTPUT_DIR / f"locust_env_{TS_UTC}.sh"
env_lines = [
    f'export OKE_VIP="{vip}"',
    f'export LOCUST_MODE="{os.environ["LOCUST_MODE"]}"',
    f'export LOCUST_VERIFY_TLS="{os.environ["LOCUST_VERIFY_TLS"]}"',
    f'export LOCUST_CONNECT_TIMEOUT_S="{os.environ["LOCUST_CONNECT_TIMEOUT_S"]}"',
    f'export LOCUST_READ_TIMEOUT_S="{os.environ["LOCUST_READ_TIMEOUT_S"]}"',
    f'export LOCUST_WAIT_TIME_S="{os.environ["LOCUST_WAIT_TIME_S"]}"',
    f'export LOCUST_HEALTH_PATH="{os.environ["LOCUST_HEALTH_PATH"]}"',
    f'export LOCUST_THROUGHPUT_PATH="{os.environ["LOCUST_THROUGHPUT_PATH"]}"',
    f'export LOCUST_TARGETS="{os.environ["LOCUST_TARGETS"]}"',
    f'export LOCUST_NAME_BY_VIP="{os.environ["LOCUST_NAME_BY_VIP"]}"',
]
env_snip_path.write_text("\n".join(env_lines) + "\n", encoding="utf-8")
print(f"Wrote: {env_snip_path}")

# 4) Validations
# 4a) HEAD to /healthz on VIP (TLS verify controlled by LOCUST_VERIFY_TLS)
curl_k = "-k" if not LOCUST_VERIFY_TLS_BOOL else ""
print("\nVIP /healthz (HEAD):")
run(f"curl --http1.1 -sS {curl_k} -I -w '%{{http_code}}\\n' --connect-timeout {CONNECT_TIMEOUT_S:.0f} --max-time {max(CONNECT_TIMEOUT_S, READ_TIMEOUT_S):.0f} {q(base_url + HEALTH_PATH)} | head -n2", check=False)

# 4b) Optional: GET “/” to surface PPv2 path body (first line)
print("\nVIP / (GET, first line; expect 'Client IP: ...' when PPv2 delivers PROXY header):")
run(f"curl --http1.1 -sS {curl_k} --connect-timeout {CONNECT_TIMEOUT_S:.0f} --max-time {max(CONNECT_TIMEOUT_S, READ_TIMEOUT_S):.0f} {q(base_url + '/')} | head -n1", check=False)

# 5) Summary
print("\nSummary:")
print(f" - VIP: {vip}")
print(f" - Targets: {', '.join(targets)}")
print(f" - Mode: {TEST_MODE}  Verify TLS: {'true' if LOCUST_VERIFY_TLS_BOOL else 'false'}")
print(f" - Timeouts: connect={CONNECT_TIMEOUT_S:.1f}s read={READ_TIMEOUT_S:.1f}s wait={WAIT_TIME_SEC:.1f}s")
print(" - Files:")
print(f"   * {vip_json_path}")
print(f"   * {targets_json_path}")
print(f"   * {env_snip_path}")
print("\nNEXT:")
print(" - If ENABLE_GENERATORS=true, proceed to Cell 9 to prepare generator hosts (pip/tmux) and distribute this env.")
print(" - Otherwise, you can run local Locust in Cell 10 using these environment variables.")


In [ ]:
# Cell 9 — Objective: Prepare Generators for Locust (robust key loading + explicit pins + reliable import check)
# - Autodetects SSH private key type (RSA/ED25519/ECDSA/DSS if available)
# - Ensures python3/pip/tmux/curl, installs pinned: requests==2.31.0 urllib3==1.26.20 locust==2.28.0 gevent==25.9.1 geventhttpclient==2.3.8
# - Uploads env snippet, ensures locustfile.py, removes legacy urllib3 shim if our marker is present
# - Verifies imports via python3 here-doc (no quoting issues), computes workers/host, HEAD /healthz from each generator
# - Writes results/generators_plan_<TS>.json; does NOT start Locust

import os, json, shlex, subprocess, time, socket
from pathlib import Path
from datetime import datetime, timezone
import paramiko

def q(s: str) -> str:
    return shlex.quote(s)

def run_local(cmd: str, check=False, quiet=False, cwd: Path|None=None):
    if not quiet: print(f"\n$ {cmd}")
    p = subprocess.run(cmd, shell=True, text=True, cwd=str(cwd) if cwd else None, capture_output=True)
    out, err = (p.stdout or "").strip(), (p.stderr or "").strip()
    if not quiet and out: print(out)
    if not quiet and err and any(w in err.lower() for w in ["error","failed","exception"]): print(err)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {cmd}\n{out}\n{err}")
    return p.returncode, out, err

def _load_stack_outputs_value(key: str):
    p = Path("stack/stack-outputs.json")
    if not p.is_file(): return None
    try:
        obj = json.loads(p.read_text(encoding="utf-8") or "{}")
        item = obj.get(key, {})
        if isinstance(item, dict) and "value" in item:
            return item.get("value")
        return item
    except Exception:
        return None

def _ms_to_s(ms_env: str, default_s: float) -> float:
    try: return max(0.0, float(int(ms_env))/1000.0)
    except Exception: return float(default_s)

def _find_latest_env_snippet(results_dir: Path) -> Path|None:
    try:
        cands = sorted(results_dir.glob("locust_env_*.sh"))
        return cands[-1] if cands else None
    except Exception:
        return None

def load_pkey_any(path: str):
    # Try available Paramiko key classes without referencing missing attributes
    order = ("RSAKey", "Ed25519Key", "ECDSAKey", "DSSKey")
    last_err = None
    for cls_name in order:
        klass = getattr(paramiko, cls_name, None)
        if not klass:
            continue
        try:
            return klass.from_private_key_file(path)
        except paramiko.ssh_exception.PasswordRequiredException as e:
            raise RuntimeError(f"Private key {path} is passphrase-protected. Provide an unencrypted key or add passphrase support.") from e
        except Exception as e:
            last_err = e
            continue
    raise last_err or RuntimeError("Unsupported or unreadable private key; tried RSA/ED25519/ECDSA/DSS.")

def ssh_connect(host: str, username: str, pkey_path: str, timeout_s: int = 25) -> paramiko.SSHClient:
    key = load_pkey_any(pkey_path)
    client = paramiko.SSHClient()
    client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    client.connect(
        hostname=host, username=username, pkey=key,
        allow_agent=False, look_for_keys=False, timeout=timeout_s,
        auth_timeout=timeout_s, banner_timeout=timeout_s
    )
    return client

def ssh_exec(client: paramiko.SSHClient, cmd: str, check: bool = False, timeout: int = 900) -> tuple[int,str,str]:
    stdin, stdout, stderr = client.exec_command(cmd, timeout=timeout)
    out = (stdout.read() or b"").decode("utf-8", "ignore").strip()
    err = (stderr.read() or b"").decode("utf-8", "ignore").strip()
    rc = stdout.channel.recv_exit_status()
    if out: print(out)
    if err and any(w in err.lower() for w in ["error","failed","exception"]): print(err)
    if check and rc != 0:
        raise RuntimeError(f"Remote command failed ({rc}): {cmd}\n{out}\n{err}")
    return rc, out, err

def ssh_sftp_put(client: paramiko.SSHClient, local_path: Path, remote_path: str):
    sftp = client.open_sftp()
    try:
        sftp.put(str(local_path), remote_path)
    finally:
        sftp.close()

def remove_shim_if_present(cli: paramiko.SSHClient):
    # Remove our previous sitecustomize shim ONLY if it clearly references urllib3 util ssl_ create_urllib3_context
    rc, usersite, _ = ssh_exec(cli, "python3 -c 'import site,sys; p=site.getusersitepackages(); print(p); sys.exit(0 if p else 1)'", check=False)
    usersite = (usersite or "").strip()
    if not usersite:
        return
    probe = f"""bash -lc 'if [ -f {q(usersite)}/sitecustomize.py ]; then grep -E "urllib3\\.util(\\.ssl_|)\\.?create_urllib3_context" -q {q(usersite)}/sitecustomize.py && echo SHIM_HIT || echo SHIM_MISS; else echo NOFILE; fi'"""
    _, mark, _ = ssh_exec(cli, probe, check=False)
    if "SHIM_HIT" in (mark or ""):
        ssh_exec(cli, f"rm -f {q(usersite)}/sitecustomize.py && echo 'removed {q(usersite)}/sitecustomize.py'", check=False)

# ---- Inputs from SSOT (Cells 2/3/8) ----
ENABLE_GENERATORS = os.environ.get("ENABLE_GENERATORS","true").strip().lower() in ("1","true","yes","y")
WORK_NS           = os.environ.get("WORK_NS", "cpstest").strip() or "cpstest"
OKE_VIP           = os.environ.get("OKE_VIP","").strip()
RESULTS_DIR       = Path(os.path.abspath(os.environ.get("OUTPUT_DIR", "./results"))); RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TS_UTC            = os.environ.get("TS_UTC", "") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"); os.environ["TS_UTC"] = TS_UTC

# Locust/env (from Cell 8)
LOCUST_MODE            = os.environ.get("LOCUST_MODE","cps").strip().lower()
LOCUST_VERIFY_TLS_BOOL = os.environ.get("LOCUST_VERIFY_TLS","false").strip().lower() in ("1","true","yes","y")
CONNECT_TIMEOUT_S      = float(os.environ.get("LOCUST_CONNECT_TIMEOUT_S", str(_ms_to_s(os.environ.get("LOCUST_CONNECT_TIMEOUT_MS","8000"), 8.0))))
READ_TIMEOUT_S         = float(os.environ.get("LOCUST_READ_TIMEOUT_S",  str(_ms_to_s(os.environ.get("LOCUST_READ_TIMEOUT_MS","15000"), 15.0))))
WAIT_TIME_S            = float(os.environ.get("LOCUST_WAIT_TIME_S", os.environ.get("LOCUST_WAIT_TIME_SEC","1.0")))
HEALTH_PATH            = os.environ.get("LOCUST_HEALTH_PATH", os.environ.get("HEALTH_ENDPOINT_PATH","/healthz")).strip() or "/healthz"
THROUGHPUT_PATH        = os.environ.get("LOCUST_THROUGHPUT_PATH", os.environ.get("THROUGHPUT_ENDPOINT_PATH","/payload_100k")).strip() or "/payload_100k"
LOCUST_TARGETS         = os.environ.get("LOCUST_TARGETS","").strip()
UI_WEB_PORT            = int(os.environ.get("UI_WEB_PORT","8089"))

# Generators SSH
SSH_PRIV = os.path.expanduser(os.environ.get("GEN_SSH_PRIVATE_KEY_PATH", os.environ.get("SSH_PRIVATE_KEY_PATH",""))).strip()
if not SSH_PRIV or not Path(SSH_PRIV).exists():
    raise FileNotFoundError(f"GEN_SSH_PRIVATE_KEY_PATH not found. Current: {SSH_PRIV or '(unset)'}")

print(f"ENABLE_GENERATORS={ENABLE_GENERATORS}  NS={WORK_NS}")
print(f"VIP={OKE_VIP or '(missing)'}  Targets={LOCUST_TARGETS or '(from env snippet)'}")
print(f"Locust: mode={LOCUST_MODE} verify_tls={'true' if LOCUST_VERIFY_TLS_BOOL else 'false'} connect={CONNECT_TIMEOUT_S}s read={READ_TIMEOUT_S}s wait={WAIT_TIME_S}s UI={UI_WEB_PORT}")
print(f"SSH key (opc): {SSH_PRIV}")

if not ENABLE_GENERATORS:
    print("ENABLE_GENERATORS=false — skipping generators prepare.")
    raise SystemExit(0)

# 1) Discover generator IPv4s
gen_ips = []
try:
    txt = os.environ.get("GEN_IPS_V4_JSON","").strip()
    if txt:
        arr = json.loads(txt)
        if isinstance(arr, list):
            gen_ips = [x for x in arr if str(x).strip()]
except Exception:
    gen_ips = []

if not gen_ips:
    fallback = _load_stack_outputs_value("gen_public_ips")
    if isinstance(fallback, list):
        gen_ips = [x for x in fallback if str(x).strip()]

if not gen_ips:
    raise SystemExit("No generator IPv4 addresses found. Ensure Cell 5/8 exported GEN_IPS_V4_JSON or stack-outputs.json exists.")

print("Generators (IPv4):", ", ".join(gen_ips))

# 2) Locate env snippet from Cell 8
env_snippet = RESULTS_DIR / f"locust_env_{TS_UTC}.sh"
if not env_snippet.exists():
    latest = _find_latest_env_snippet(RESULTS_DIR)
    if latest and latest.exists():
        env_snippet = latest
if not env_snippet.exists():
    raise SystemExit(f"Env snippet not found. Expected: {RESULTS_DIR}/locust_env_{TS_UTC}.sh or latest in {RESULTS_DIR}")
print(f"Using env snippet: {env_snippet}")

# 3) Prepare each generator
prep_results = []
master_ip = gen_ips[0]
LOCUST_WORKDIR = "/home/opc/locustwork"

DEFAULT_LOCUSTFILE = r"""import os
from itertools import cycle
from locust import HttpUser, task, constant

VERIFY_TLS        = os.environ.get("LOCUST_VERIFY_TLS", "false").lower() == "true"
CONNECT_TIMEOUT_S = float(os.environ.get("LOCUST_CONNECT_TIMEOUT_S", "8"))
READ_TIMEOUT_S    = float(os.environ.get("LOCUST_READ_TIMEOUT_S", "15"))
WAIT_TIME_S       = float(os.environ.get("LOCUST_WAIT_TIME_S", "1.0"))
MODE              = os.environ.get("LOCUST_MODE", "cps").lower()
HEALTH_PATH       = os.environ.get("LOCUST_HEALTH_PATH", "/healthz")
THROUGHPUT_PATH   = os.environ.get("LOCUST_THROUGHPUT_PATH", "/payload_100k")
RAW_TARGETS       = os.environ.get("LOCUST_TARGETS", "")
TARGETS           = [t.strip() for t in RAW_TARGETS.split(",") if t.strip()]
NAME_BY_VIP       = os.environ.get("LOCUST_NAME_BY_VIP", "true").lower() == "true"

class CpsUser(HttpUser):
    host = os.environ.get("LOCUST_DEFAULT_HOST", "https://127.0.0.1")
    wait_time = constant(WAIT_TIME_S)
    def on_start(self):
        self._targets = cycle(TARGETS) if TARGETS else None
    def _next_base(self):
        try:
            return next(self._targets) if self._targets else (self.host or "")
        except Exception:
            return self.host or ""
    @task
    def do_request(self):
        path = HEALTH_PATH if MODE == "cps" else THROUGHPUT_PATH
        base = self._next_base()
        url  = f"{base}{path}" if base else path
        headers = {"Connection": "close"} if MODE == "cps" else {}
        name_val = path
        if NAME_BY_VIP and base:
            vip = base.replace("https://","").replace("http://","")
            name_val = f"{vip}{path}"
        self.client.get(url, headers=headers,
                        verify=VERIFY_TLS,
                        timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S),
                        name=name_val)
"""

def ensure_python_and_tools(cli: paramiko.SSHClient):
    ssh_exec(cli, "command -v python3 || (sudo -n dnf -y install python3 || sudo -n yum -y install python3)", check=False)
    ssh_exec(cli, "sudo -n dnf -y install python3-pip tmux curl || sudo -n yum -y install python3-pip tmux curl", check=False)
    ssh_exec(cli, "python3 -m pip install --user --upgrade --no-cache-dir pip setuptools wheel", check=False)

def install_pins(cli: paramiko.SSHClient):
    # Explicit pins to avoid long resolver backtracking and ensure Python 3.12 compatibility on OL10
    pin_cmd = (
        'export PATH="$HOME/.local/bin:$PATH"; '
        'python3 -m pip install --user --upgrade --no-cache-dir '
        '"requests==2.31.0" "urllib3==1.26.20" "locust==2.28.0" "gevent==25.9.1" "geventhttpclient==2.3.8"'
    )
    ssh_exec(cli, pin_cmd, check=True)
    # Robust import check via here-doc (quotes preserved)
    import_check = r"""bash -lc 'export PATH="$HOME/.local/bin:$PATH"; python3 - <<'\''PY'\''
import locust, gevent, geventhttpclient, requests, urllib3
print("OK")
PY
'"""
    ssh_exec(cli, import_check, check=True)

def prepare_host(host: str):
    print(f"\n=== Preparing generator: {host} ===")
    try:
        cli = ssh_connect(host, "opc", SSH_PRIV, timeout_s=25)
        try:
            ensure_python_and_tools(cli)
            remove_shim_if_present(cli)

            # Workspace and env
            ssh_exec(cli, f"mkdir -p {q(LOCUST_WORKDIR)} && chmod 755 {q(LOCUST_WORKDIR)}", check=True)
            ssh_sftp_put(cli, env_snippet, f"{LOCUST_WORKDIR}/.env")
            ssh_exec(cli, f"chown opc:opc {q(LOCUST_WORKDIR)}/.env && chmod 600 {q(LOCUST_WORKDIR)}/.env", check=False)

            # PATH + auto-source .env in login shells
            bashrc_fix = r"""if ! grep -q 'locustwork/.env' ~/.bashrc 2>/dev/null; then
  echo 'export PATH="$HOME/.local/bin:$PATH"' >> ~/.bashrc
  echo 'if [ -f /home/opc/locustwork/.env ]; then . /home/opc/locustwork/.env; fi' >> ~/.bashrc
fi"""
            ssh_exec(cli, f"bash -lc {q(bashrc_fix)}", check=False)

            # Install pins and verify imports
            install_pins(cli)

            # Ensure locustfile.py exists
            rc, present, _ = ssh_exec(cli, f"test -f {q(LOCUST_WORKDIR)}/locustfile.py && echo PRESENT || echo MISSING", check=False)
            if "MISSING" in (present or ""):
                mk_locust = f"bash -lc {q(f'cat > {LOCUST_WORKDIR}/locustfile.py <<\"PY\"\n{DEFAULT_LOCUSTFILE}\nPY\nchmod 0644 {LOCUST_WORKDIR}/locustfile.py')}"
                ssh_exec(cli, mk_locust, check=True)

            # Discover CPU count (for auto policy)
            rc, nproc_out, _ = ssh_exec(cli, "nproc || getconf _NPROCESSORS_ONLN || echo 1", check=False)
            try:
                nproc = max(1, int((nproc_out or "1").strip().splitlines()[0]))
            except Exception:
                nproc = 1

            return cli, nproc
        except Exception:
            raise
    except (paramiko.ssh_exception.SSHException, socket.timeout, Exception) as e:
        print(f"[WARN] Generator {host} prepare failed: {e!r}")
        return None, None

# Workers policy (Cells 2/3)
WORKERS_PER_HOST_ENV = os.environ.get("WORKERS_PER_HOST","auto").strip().lower() or "auto"
CPU_RESERVE          = int(os.environ.get("CPU_RESERVE","1"))
MIN_WORKERS          = int(os.environ.get("MIN_WORKERS_PER_HOST","1"))
MAX_WORKERS          = int(os.environ.get("MAX_WORKERS_PER_HOST","32"))

print(f"Workers policy: {WORKERS_PER_HOST_ENV} (reserve={CPU_RESERVE} min={MIN_WORKERS} max={MAX_WORKERS})")

prep_results = []
live_clients = {}
master_ip = gen_ips[0]
for host in gen_ips:
    cli, nproc = prepare_host(host)
    if not cli:
        prep_results.append({"ip": host, "error": "prepare_failed"})
        continue

    # Compute workers/host
    if WORKERS_PER_HOST_ENV == "auto":
        effective = max(1, (nproc or 1) - max(0, CPU_RESERVE))
        bounded  = min(MAX_WORKERS, effective)
        workers_host = max(MIN_WORKERS, bounded)
    else:
        try:
            fixed = max(1, int(WORKERS_PER_HOST_ENV))
        except Exception:
            fixed = 1
        workers_host = fixed

    # Validate VIP reachability from generator (HEAD /healthz)
    curl_k = "-k" if (not LOCUST_VERIFY_TLS_BOOL) else ""
    rc, head_out, _ = ssh_exec(
        cli,
        f"bash -lc 'source {q(LOCUST_WORKDIR)}/.env >/dev/null 2>&1 || true; "
        f"curl --http1.1 -sS {curl_k} -I -w \"%{{http_code}}\\n\" --connect-timeout {int(CONNECT_TIMEOUT_S)} "
        f"--max-time {int(max(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))} {q(f'https://{OKE_VIP}{HEALTH_PATH}')} | head -n2'",
        check=False
    )

    prep_results.append({
        "ip": host,
        "nproc": nproc or 1,
        "workers_planned": workers_host,
        "vip_health_head": head_out.splitlines() if head_out else [],
    })
    live_clients[host] = cli
    print(f"Prepared {host}: nproc={nproc or 1} workers_planned={workers_host}")

# Close SSH clients
for h, c in list(live_clients.items()):
    try: c.close()
    except Exception: pass

# 4) Plan artifact
plan = {
    "ts_utc": TS_UTC,
    "vip": OKE_VIP,
    "targets": LOCUST_TARGETS,
    "mode": LOCUST_MODE,
    "verify_tls": LOCUST_VERIFY_TLS_BOOL,
    "connect_timeout_s": CONNECT_TIMEOUT_S,
    "read_timeout_s": READ_TIMEOUT_S,
    "wait_time_s": WAIT_TIME_S,
    "ui_web_port": UI_WEB_PORT,
    "workers_policy": {
        "mode": WORKERS_PER_HOST_ENV,
        "cpu_reserve": CPU_RESERVE,
        "min_per_host": MIN_WORKERS,
        "max_per_host": MAX_WORKERS
    },
    "master_ip": master_ip,
    "generators": prep_results
}
plan_path = RESULTS_DIR / f"generators_plan_{TS_UTC}.json"
plan_path.write_text(json.dumps(plan, indent=2), encoding="utf-8")
print(f"\nWrote plan: {plan_path}")

# 5) Summary
ok_count = sum(1 for r in prep_results if not r.get("error"))
print("\nSummary:")
print(f" - Generators: {len(gen_ips)} total, prepared_ok={ok_count}")
print(f" - Master IP: {master_ip}")
for r in prep_results:
    ip = r.get("ip")
    if "error" in r:
        print(f"   * {ip}: ERROR: {r['error']}")
    else:
        code_line = ""
        try:
            lines = r.get("vip_health_head") or []
            code_line = lines[-1] if lines else ""
        except Exception:
            code_line = ""
        print(f"   * {ip}: nproc={r['nproc']} workers_planned={r['workers_planned']} health_code={code_line}")

print("\nNEXT (Cell 10):")
print(" - Start Locust: master on the master IP and workers on all generators (including master) using the prepared plan/env.")
print(f" - Locust UI will be at: http://{master_ip}:{UI_WEB_PORT} (once started).")


In [ ]:
# Cell 10 — Objective: Start Locust master (tmux) + workers (tmux, paced) with explicit RPC port and readiness checks
# - Stops existing UI/workers/tmux
# - Starts master on 0.0.0.0:8089 and RPC 0.0.0.0:5557 (tmux ui_master)
# - Spawns N workers/host under tmux session "wkr" with small delay between spawns
# - Verifies listeners & worker counts; tails newest worker log on non-ready hosts

import os, json, shlex, time
from pathlib import Path
import paramiko

def q(s: str) -> str:
    return shlex.quote(s)

RESULTS_DIR = Path(os.path.abspath(os.environ.get("OUTPUT_DIR", "./results")))
ts = os.environ.get("TS_UTC","")
plan_path = RESULTS_DIR / f"generators_plan_{ts}.json" if ts else None
if not plan_path or not plan_path.exists():
    cands = sorted(RESULTS_DIR.glob("generators_plan_*.json"))
    plan_path = cands[-1] if cands else None
if not plan_path or not plan_path.exists():
    raise SystemExit("Plan file not found. Run Cell 9 first to create results/generators_plan_<TS>.json.")

plan = json.loads(plan_path.read_text(encoding="utf-8") or "{}")
master_ip = plan.get("master_ip") or ""
ui_port   = int(plan.get("ui_web_port", 8089))
gens      = [g.get("ip") for g in (plan.get("generators") or []) if g.get("ip")]
if not (master_ip and gens):
    raise SystemExit("Plan missing master_ip or generators.")

SSH_PRIV = os.path.expanduser(os.environ.get("GEN_SSH_PRIVATE_KEY_PATH", os.environ.get("SSH_PRIVATE_KEY_PATH",""))).strip()
if not SSH_PRIV or not Path(SSH_PRIV).exists():
    raise FileNotFoundError(f"GEN_SSH_PRIVATE_KEY_PATH not found. Current: {SSH_PRIV or '(unset)'}")

LOCUST_WORKDIR   = "/home/opc/locustwork"
MASTER_RPC_PORT  = 5557
SPAWN_SLEEP      = 0.15   # seconds between worker starts
SESSION_NAME     = "wkr"  # tmux session name for workers

def load_key():
    for cls in ("RSAKey","Ed25519Key","ECDSAKey","DSSKey"):
        k = getattr(paramiko, cls, None)
        if not k: continue
        try:
            return k.from_private_key_file(SSH_PRIV)
        except Exception:
            continue
    raise RuntimeError("Could not load private key in any supported format.")
PKEY = load_key()

def ssh_conn(host: str) -> paramiko.SSHClient:
    c = paramiko.SSHClient()
    c.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    c.connect(hostname=host, username="opc", pkey=PKEY, allow_agent=False, look_for_keys=False, timeout=25,
              auth_timeout=25, banner_timeout=25)
    return c

def rexec(c: paramiko.SSHClient, cmd: str, timeout=120):
    _, out, err = c.exec_command(cmd, timeout=timeout)
    out = (out.read() or b"").decode("utf-8","ignore").strip()
    err = (err.read() or b"").decode("utf-8","ignore").strip()
    return out, err

# 1) Stop existing UI/workers/tmux
print("Stopping existing UI/workers ...")
for idx, host in enumerate([master_ip] + [h for h in gens if h != master_ip]):
    c = ssh_conn(host)
    try:
        rexec(c, f"tmux has-session -t {q(SESSION_NAME)} 2>/dev/null && tmux kill-session -t {q(SESSION_NAME)} || true", timeout=20)
        if idx == 0:
            rexec(c, "tmux has-session -t ui_master 2>/dev/null && tmux kill-session -t ui_master || true", timeout=20)
        rexec(c, r'pkill -f "python3 -m locust .*--worker" 2>/dev/null || true', timeout=20)
        rexec(c, r'pkill -f "locust .*--worker" 2>/dev/null || true', timeout=20)
    finally:
        c.close()
print("Stopped.")

# 2) Start master (explicit RPC port) in tmux ui_master
print(f"\nStarting master on {master_ip}: UI {ui_port}, RPC {MASTER_RPC_PORT} ...")
master_script = f"""#!/usr/bin/env bash
set -euo pipefail
export PATH="$HOME/.local/bin:/usr/local/bin:/usr/bin:/bin:$PATH"
if [ -f "{LOCUST_WORKDIR}/.env" ]; then . "{LOCUST_WORKDIR}/.env"; fi
cd {LOCUST_WORKDIR}
mkdir -p logs
tmux has-session -t ui_master 2>/dev/null && tmux kill-session -t ui_master || true
tmux new -d -s ui_master bash -lc 'exec python3 -m locust -f {LOCUST_WORKDIR}/locustfile.py --master --master-bind-host 0.0.0.0 --master-bind-port {MASTER_RPC_PORT} --web-host 0.0.0.0 --web-port {ui_port} > {LOCUST_WORKDIR}/logs/master_ui.log 2>&1'
"""
c = ssh_conn(master_ip)
try:
    sftp = c.open_sftp()
    try:
        with sftp.file(f"{LOCUST_WORKDIR}/run_ui_fix.sh", "w") as f:
            f.write(master_script)
        sftp.chmod(f"{LOCUST_WORKDIR}/run_ui_fix.sh", 0o755)
    finally:
        sftp.close()
    rexec(c, f"bash -lc {q(f'{LOCUST_WORKDIR}/run_ui_fix.sh')}", timeout=30)
finally:
    c.close()

c = ssh_conn(master_ip)
try:
    print("\n[MASTER] Listening (expect 8089 and 5557):")
    out,_ = rexec(c, r"ss -lntp | egrep ':(5557|8089)\s' || true", timeout=20)
    print(out or "(none)")
finally:
    c.close()

# 3) Determine per-host worker count from policy
def detect_nproc(host: str) -> int:
    c = ssh_conn(host)
    try:
        out,_ = rexec(c, "nproc 2>/dev/null || getconf _NPROCESSORS_ONLN 2>/dev/null || echo 1", timeout=15)
        try: return max(1, int((out or "1").strip().splitlines()[0]))
        except Exception: return 1
    finally:
        c.close()

WORKERS_PER_HOST_ENV = str(plan.get("workers_policy", {}).get("mode", "auto")).strip().lower()
CPU_RESERVE          = int(plan.get("workers_policy", {}).get("cpu_reserve", 1))
MIN_WORKERS          = int(plan.get("workers_policy", {}).get("min_per_host", 1))
MAX_WORKERS          = int(plan.get("workers_policy", {}).get("max_per_host", 32))

def calc_workers(host: str) -> int:
    if WORKERS_PER_HOST_ENV == "auto":
        n = detect_nproc(host)
        eff = max(1, n - max(0, CPU_RESERVE))
        return max(MIN_WORKERS, min(MAX_WORKERS, eff))
    else:
        try: return max(1, int(WORKERS_PER_HOST_ENV))
        except Exception: return 1

all_hosts = [master_ip] + [x for x in gens if x != master_ip]
per_host = {h: calc_workers(h) for h in all_hosts}
print("\nPlanned workers per host:", per_host, "| total =", sum(per_host.values()))

# 4) Upload spawn and launcher scripts, then start tmux per host
def upload_and_launch(host: str, total: int):
    spawn_script = f"""#!/usr/bin/env bash
set -euo pipefail
export PATH="$HOME/.local/bin:/usr/local/bin:/usr/bin:/bin:$PATH"
if [ -f "{LOCUST_WORKDIR}/.env" ]; then . "{LOCUST_WORKDIR}/.env"; fi
cd "{LOCUST_WORKDIR}"
mkdir -p logs
echo "Spawning {total} worker(s) to master {master_ip}:{MASTER_RPC_PORT} at $(date -u)" | tee -a logs/spawn_tmux.out
i=0
while [ "$i" -lt "{total}" ]; do
  i=$((i+1))
  python3 -m locust -f locustfile.py --worker --master-host {master_ip} --master-port {MASTER_RPC_PORT} > "logs/worker-$i.log" 2>&1 &
  sleep {SPAWN_SLEEP}
done
echo "Workers started: {total} at $(date -u)" | tee -a logs/spawn_tmux.out
tail -f /dev/null
"""
    launcher_script = f"""#!/usr/bin/env bash
set -euo pipefail
export PATH="$HOME/.local/bin:/usr/local/bin:/usr/bin:/bin:$PATH"
cd "{LOCUST_WORKDIR}"
tmux has-session -t {SESSION_NAME} 2>/dev/null && tmux kill-session -t {SESSION_NAME} || true
tmux new -d -s {SESSION_NAME} bash -lc '{LOCUST_WORKDIR}/spawn_workers_tmux.sh'
"""

    c = ssh_conn(host)
    try:
        sftp = c.open_sftp()
        try:
            with sftp.file(f"{LOCUST_WORKDIR}/spawn_workers_tmux.sh", "w") as f:
                f.write(spawn_script)
            sftp.chmod(f"{LOCUST_WORKDIR}/spawn_workers_tmux.sh", 0o755)

            with sftp.file(f"{LOCUST_WORKDIR}/run_tmux_launcher.sh", "w") as f:
                f.write(launcher_script)
            sftp.chmod(f"{LOCUST_WORKDIR}/run_tmux_launcher.sh", 0o755)
        finally:
            sftp.close()
        rexec(c, f"bash -lc {q(f'{LOCUST_WORKDIR}/run_tmux_launcher.sh')}", timeout=20)
        print(f"  {host}: tmux session {SESSION_NAME} launched")
    finally:
        c.close()

print("\nLaunching tmux-based workers ...")
for h, n in per_host.items():
    upload_and_launch(h, n)

# 5) Readiness after settle
def count_workers(host: str) -> int:
    c = ssh_conn(host)
    try:
        out,_ = rexec(c, r"pgrep -f 'locust.*--worker' | wc -l || true", timeout=10)
        try: return int((out or "0").strip())
        except Exception: return 0
    finally:
        c.close()

time.sleep(20)

print("\nReadiness per host:")
for h, expected in per_host.items():
    seen = count_workers(h)
    ok = seen >= expected
    print(f"  {h}: {'READY' if ok else 'NOT READY'} (seen={seen}, expected~{expected})")
    if not ok:
        c = ssh_conn(h)
        try:
            out,_ = rexec(c, r'ls -1t "$HOME/locustwork"/logs/worker-*.log 2>/dev/null | head -n1 | xargs -r tail -n 80 || true', timeout=20)
            if out:
                print("    worker log (tail):")
                print("    " + "\n    ".join(out.splitlines()))
        finally:
            c.close()

print(f"\nLocust UI: http://{master_ip}:{ui_port}")
print("Use the UI to start tests once workers show READY.")


In [ ]:
# Cell 11 — Objective: Stop Locust UI and ALL workers (tmux-aware) and verify zero remain

import os, json
from pathlib import Path
import paramiko, shlex, time

def q(s: str) -> str:
    return shlex.quote(s)

RESULTS_DIR = Path(os.path.abspath(os.environ.get("OUTPUT_DIR", "./results")))
ts = os.environ.get("TS_UTC","")
plan_path = RESULTS_DIR / f"generators_plan_{ts}.json" if ts else None
if not plan_path or not plan_path.exists():
    cands = sorted(RESULTS_DIR.glob("generators_plan_*.json"))
    plan_path = cands[-1] if cands else None
if not plan_path or not plan_path.exists():
    raise SystemExit("Plan file not found. Run Cell 9 first to create results/generators_plan_<TS>.json.")

plan = json.loads(plan_path.read_text(encoding="utf-8") or "{}")
master_ip = plan.get("master_ip") or ""
gens      = [g.get("ip") for g in (plan.get("generators") or []) if g.get("ip")]
if not (master_ip and gens):
    raise SystemExit("Plan missing master_ip or generators.")

SSH_PRIV = os.path.expanduser(os.environ.get("GEN_SSH_PRIVATE_KEY_PATH", os.environ.get("SSH_PRIVATE_KEY_PATH",""))).strip()
if not SSH_PRIV or not Path(SSH_PRIV).exists():
    raise FileNotFoundError(f"GEN_SSH_PRIVATE_KEY_PATH not found. Current: {SSH_PRIV or '(unset)'}")

LOCUST_WORKDIR = "/home/opc/locustwork"
SESSION_NAME   = "wkr"

def load_key():
    for cls in ("RSAKey","Ed25519Key","ECDSAKey","DSSKey"):
        k = getattr(paramiko, cls, None)
        if not k: continue
        try:
            return k.from_private_key_file(SSH_PRIV)
        except Exception:
            continue
    raise RuntimeError("Could not load private key in any supported format.")
PKEY = load_key()

def ssh_conn(host: str) -> paramiko.SSHClient:
    c = paramiko.SSHClient()
    c.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    c.connect(hostname=host, username="opc", pkey=PKEY, allow_agent=False, look_for_keys=False, timeout=25,
              auth_timeout=25, banner_timeout=25)
    return c

def rexec(c: paramiko.SSHClient, cmd: str, timeout=90):
    _, out, err = c.exec_command(cmd, timeout=timeout)
    out = (out.read() or b"").decode("utf-8","ignore").strip()
    err = (err.read() or b"").decode("utf-8","ignore").strip()
    return out, err

# 1) Stop UI and per-host worker tmux sessions
print("Stopping Locust UI (tmux ui_master) and workers (tmux wkr) ...")
for idx, host in enumerate([master_ip] + [h for h in gens if h != master_ip]):
    c = ssh_conn(host)
    try:
        if idx == 0:
            rexec(c, "tmux has-session -t ui_master 2>/dev/null && tmux kill-session -t ui_master || true", timeout=20)
        rexec(c, f"tmux has-session -t {q(SESSION_NAME)} 2>/dev/null && tmux kill-session -t {q(SESSION_NAME)} || true", timeout=20)
        # Also kill any stray workers, just in case
        rexec(c, r'pkill -f "python3 -m locust .*--worker" 2>/dev/null || true', timeout=20)
        rexec(c, r'pkill -f "locust .*--worker" 2>/dev/null || true', timeout=20)
    finally:
        c.close()
print("Stop requested.")

# 2) Verify zero worker processes (with a brief retry)
def count_workers(host: str) -> int:
    c = ssh_conn(host)
    try:
        out,_ = rexec(c, r"pgrep -f 'locust.*--worker' | wc -l || true", timeout=10)
        try: return int((out or '0').strip())
        except Exception: return 0
    finally:
        c.close()

time.sleep(2)
print("\nVerification (workers per host):")
remaining = {}
for host in [master_ip] + [h for h in gens if h != master_ip]:
    cnt = count_workers(host)
    print(f"  {host}: workers={cnt}")
    if cnt > 0:
        remaining[host] = cnt

if remaining:
    print("\nWARNING: Some hosts still show worker processes. Run this cell again in a few seconds to re-issue kills, or SSH to inspect:")
    for h,c in remaining.items():
        print(f"  ssh -i {SSH_PRIV} opc@{h} 'pgrep -a -f \"locust.*--worker\"; ps -ef | grep -E \"locust .*--worker|python3 -m locust\"'")
else:
    print("\nAll workers stopped successfully on all hosts.")


In [ ]:
# Cell 12 — Objective: Optional cleanup of generator workspaces (no processes touched)
# - Removes Locust logs and spawn outputs on all hosts (including master)
# - Keeps .env and locustfile.py intact so you can re-run Cell 10 immediately
# - Idempotent; safe to skip if you want to retain logs

import os, json
from pathlib import Path
import paramiko

def q(s: str) -> str:
    return s.replace('"', '\\"')

# Locate plan from Cell 9/10
RESULTS_DIR = Path(os.path.abspath(os.environ.get("OUTPUT_DIR", "./results")))
TS_UTC = os.environ.get("TS_UTC", "")
plan_path = RESULTS_DIR / f"generators_plan_{TS_UTC}.json" if TS_UTC else None
if not plan_path or not plan_path.exists():
    cands = sorted(RESULTS_DIR.glob("generators_plan_*.json"))
    plan_path = (cands[-1] if cands else None)
if not plan_path or not plan_path.exists():
    raise SystemExit("Generators plan not found. Run Cell 9 first to create results/generators_plan_<TS>.json.")

plan = json.loads(plan_path.read_text(encoding="utf-8") or "{}")
master_ip = plan.get("master_ip") or ""
gens      = plan.get("generators") or []
if not master_ip or not gens:
    raise SystemExit("Plan missing master_ip or generators.")

SSH_PRIV = os.path.expanduser(os.environ.get("GEN_SSH_PRIVATE_KEY_PATH", os.environ.get("SSH_PRIVATE_KEY_PATH",""))).strip()
if not SSH_PRIV or not Path(SSH_PRIV).exists():
    raise FileNotFoundError(f"GEN_SSH_PRIVATE_KEY_PATH not found. Current: {SSH_PRIV or '(unset)'}")

LOCUST_WORKDIR = "/home/opc/locustwork"

# Build host list with master first
all_hosts = [h.get("ip") for h in gens if h.get("ip")]
if master_ip not in all_hosts:
    all_hosts = [master_ip] + all_hosts
else:
    all_hosts = [master_ip] + [h for h in all_hosts if h != master_ip]

def ssh_connect(host: str) -> paramiko.SSHClient:
    key = paramiko.RSAKey.from_private_key_file(SSH_PRIV)
    cli = paramiko.SSHClient()
    cli.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cli.connect(hostname=host, username="opc", pkey=key, allow_agent=False, look_for_keys=False, timeout=25)
    return cli

def ssh_exec(cli: paramiko.SSHClient, cmd: str, timeout=120):
    stdin, stdout, stderr = cli.exec_command(cmd, timeout=timeout)
    out = (stdout.read() or b"").decode("utf-8","ignore").strip()
    err = (stderr.read() or b"").decode("utf-8","ignore").strip()
    rc = stdout.channel.recv_exit_status()
    return rc, out, err

CLEAN_PATTERNS = ["worker-*.log", "spawn.out", "spawn_master.out", "master_ui.log"]

print("Cleaning Locust logs on all hosts (keeping .env and locustfile.py) ...")
for host in all_hosts:
    try:
        cli = ssh_connect(host)
        try:
            # Ensure logs dir exists (harmless if missing)
            ssh_exec(cli, f'mkdir -p "{q(LOCUST_WORKDIR)}"/logs', timeout=10)
            # Remove only log artifacts
            for pat in CLEAN_PATTERNS:
                ssh_exec(cli, f'rm -f "{q(LOCUST_WORKDIR)}"/logs/{pat} 2>/dev/null || true', timeout=15)
            print(f"  {host}: cleaned")
        finally:
            cli.close()
    except Exception as e:
        print(f"  {host}: cleanup error {e!r}")

print("Cell 12 complete: Logs cleaned. Re-run Cell 10 to start a fresh run, or stop here.")


In [ ]:
# Cell 13 — Objective: Full Teardown (clean order, no hard failures)
# - K8s: delete Service first (to trigger CCM LB teardown), wait for LB removal (or force-delete), then delete other K8s objects
# - Optional: delete namespace and wait for termination (guarded by TEARDOWN_DELETE_NAMESPACE)
# - OCI: terraform destroy the stack (retry once if needed); no unsupported node-pool size edits
# - oci CLI is optional; if present and env is set, used for LB polling and informational checks
# - Idempotent and defensive: avoids raising exceptions; prints status instead

import os, json, shlex, subprocess, time, re
from pathlib import Path

# ---------- Helpers ----------
def run(cmd: str, cwd: Path|None=None, echo=True) -> tuple[int, str, str]:
    if echo:
        print(f"\n$ {cmd}")
    p = subprocess.run(cmd, shell=True, text=True, cwd=str(cwd) if cwd else None,
                       capture_output=True)
    out, err = (p.stdout or "").strip(), (p.stderr or "").strip()
    if echo and out:
        print(out)
    # only print stderr if it looks like an error to avoid noise
    if echo and err and any(w in err.lower() for w in ["error","failed","exception"]):
        print(err)
    return p.returncode, out, err

def which(name: str) -> bool:
    try:
        import shutil
        return shutil.which(name) is not None
    except Exception:
        return False

def derive_region_from_cluster_ocid() -> str:
    p = Path("stack/stack-outputs.json")
    if not p.is_file(): return ""
    try:
        obj = json.loads(p.read_text(encoding="utf-8") or "{}")
        cid = obj.get("cluster_id", {}); cid = cid.get("value") if isinstance(cid, dict) else cid
        m = re.search(r"^ocid1\.cluster\.oc1\.([^.]+)\.", (cid or ""))
        return m.group(1) if m else ""
    except Exception: return ""

def find_lb_id_by_vip(vip: str, comp: str, region: str) -> str:
    if not (vip and comp and region and which("oci")):
        return ""
    rc, out, _ = run(
        f"oci lb load-balancer list --compartment-id {shlex.quote(comp)} --all "
        f"--region {shlex.quote(region)} --output json 2>/dev/null",
        echo=False
    )
    if rc != 0 or not out:
        return ""
    try:
        data = json.loads(out).get("data") or []
        for item in data:
            for addr in (item.get("ip-addresses") or []):
                if (addr.get("ip-address") or "") == vip:
                    return item.get("id") or ""
    except Exception:
        return ""
    return ""

# ---------- Inputs & toggles ----------
NS           = os.environ.get("WORK_NS", "cpstest").strip() or "cpstest"
SVC_NAME     = os.environ.get("SVC_NAME", "nginx-service").strip() or "nginx-service"
REGION       = os.environ.get("REGION", "").strip()
COMP         = os.environ.get("COMPARTMENT_ID", "").strip()
OKE_VIP      = os.environ.get("OKE_VIP", "").strip()
DELETE_NS    = os.environ.get("TEARDOWN_DELETE_NAMESPACE", "false").strip().lower() in ("1","true","yes","y")
FORCE_DEL_LB = os.environ.get("TEARDOWN_FORCE_DELETE_LB", "false").strip().lower() in ("1","true","yes","y")

print(f"Teardown start: ns={NS}, svc={SVC_NAME}, region={REGION or '(unset)'}, compartment={COMP or '(unset)'}")
print(f"Options: delete_namespace={'true' if DELETE_NS else 'false'}, force_delete_lb={'true' if FORCE_DEL_LB else 'false'}")

# ---------- A) K8s ordered teardown ----------
if which("kubectl"):
    print("\nK8s teardown (ordered): delete Service → wait LB gone → delete rest → optionally delete ns ...")

    # 1) Delete Service first (triggers CCM LB teardown)
    run(f"kubectl -n {shlex.quote(NS)} delete svc {shlex.quote(SVC_NAME)} --ignore-not-found", echo=True)

    # 2) Best-effort wait for OCI LB VIP to disappear BEFORE touching the namespace
    if which("oci"):
        if not REGION:
            rg = derive_region_from_cluster_ocid()
            if rg:
                REGION = rg
                print(f"Derived REGION from cluster ocid: {REGION}")

        if OKE_VIP and COMP and REGION:
            print(f"\nPolling for LB deletion (VIP={OKE_VIP}) up to 300s ...")
            end = time.time() + 300
            lb_id = ""
            while time.time() < end:
                lb_id = find_lb_id_by_vip(OKE_VIP, COMP, REGION)
                if not lb_id:
                    print("LB not found (VIP gone).")
                    break
                print("LB still present; waiting ...")
                time.sleep(5)
            # Optional: force delete if requested and LB remains
            if lb_id and FORCE_DEL_LB:
                print(f"Force deleting LB {lb_id} (best-effort) ...")
                run(
                    f"oci lb load-balancer delete --load-balancer-id {shlex.quote(lb_id)} "
                    f"--force --wait-for-state SUCCEEDED --region {shlex.quote(REGION)}",
                    echo=True
                )
        else:
            print("Skipping LB poll (missing VIP/COMP/REGION).")
    else:
        print("oci CLI not found — skipping LB poll and relying on CCM to complete teardown in background.")

    # 3) Delete remaining workload objects (DaemonSets, Deployments, ConfigMap, TLS secret)
    run(f"kubectl -n {shlex.quote(NS)} delete deploy nginx-backend-extra --ignore-not-found", echo=True)
    run(f"kubectl -n {shlex.quote(NS)} delete ds nginx-backend --ignore-not-found", echo=True)
    run(f"kubectl -n {shlex.quote(NS)} delete ds ds-backend-tuner --ignore-not-found", echo=True)
    run(f"kubectl -n {shlex.quote(NS)} delete cm nginx-config --ignore-not-found", echo=True)
    run(f"kubectl -n {shlex.quote(NS)} delete secret ssl-certificate-secret --ignore-not-found", echo=True)

    # 4) Optionally delete namespace and wait for it to finish terminating
    if DELETE_NS:
        print(f"\nDeleting namespace {NS} and waiting up to 180s for termination ...")
        run(f"kubectl delete ns {shlex.quote(NS)} --ignore-not-found", echo=True)
        # Wait until kubectl get ns returns non-zero (gone) or phase != Terminating
        end = time.time() + 180
        while time.time() < end:
            rc, out, _ = run(
                f"kubectl get ns {shlex.quote(NS)} -o jsonpath='{{.status.phase}}'",
                echo=False
            )
            if rc != 0:
                print("Namespace removed.")
                break
            # if still present, print occasional heartbeat
            time.sleep(3)
        else:
            print("Namespace still terminating after 180s (may complete shortly).")
else:
    print("kubectl not found — skipping K8s object deletion.")

# ---------- B) Terraform destroy ----------
stack_dir = Path("stack").resolve()
if not stack_dir.is_dir():
    print("\nTerraform stack directory not found; skipping destroy of stack/ (already removed?).")
else:
    print("\nTerraform destroy: deprovision everything under ./stack (idempotent) ...")

    # Ensure init (helps when .terraform is stale or lock missing)
    run("terraform init -upgrade -input=false", cwd=stack_dir, echo=True)

    # First attempt
    rc, _, _ = run("terraform destroy -parallelism=20 -auto-approve -var-file=terraform.tfvars",
                   cwd=stack_dir, echo=True)
    if rc != 0:
        print("Destroy reported non-zero exit; retrying once after 10s ...")
        time.sleep(10)
        run("terraform init -upgrade -input=false", cwd=stack_dir, echo=True)
        rc2, _, _ = run("terraform destroy -parallelism=20 -auto-approve -var-file=terraform.tfvars",
                        cwd=stack_dir, echo=True)
        if rc2 != 0:
            print("Final destroy attempt also reported non-zero exit. Check above logs; some resources may require manual cleanup.")

# ---------- C) Summary ----------
print("\nTeardown complete (best-effort):")
print(f" - K8s objects removed in ns/{NS}" + (" and namespace deletion requested" if DELETE_NS else ""))
if which("oci") and OKE_VIP and COMP and REGION:
    lb_check = find_lb_id_by_vip(OKE_VIP, COMP, REGION)
    print(f" - LB VIP {OKE_VIP} present after teardown? {'YES: '+lb_check if lb_check else 'NO'}")
else:
    print(" - Skipped LB presence check (oci not available or VIP/COMP/REGION unset).")
print(" - Terraform stack destroy attempted (with a single retry on failure).")
print("\nYou can now re-run Cell 4 (to generate a fresh stack) followed by Cell 5 (apply) when you want to redeploy.")
